In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["TF_NUM_INTRAOP_THREADS"] = "2"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import tensorflow as tf


def configure_available_gpus():
    """
    Configure all visible GPUs once in the notebook's main process.

    This function must be called before TensorFlow initializes
    either GPU.
    """

    physical_gpus = tf.config.list_physical_devices("GPU")

    if len(physical_gpus) == 0:
        raise RuntimeError(
            "TensorFlow could not find any GPU."
        )

    for physical_gpu in physical_gpus:
        try:
            tf.config.experimental.set_memory_growth(
                physical_gpu,
                True
            )
        except RuntimeError as error:
            raise RuntimeError(
                "GPU configuration has already been initialized. "
                "Restart the notebook kernel and run this cell before "
                "executing TensorFlow operations."
            ) from error

    tf.config.threading.set_intra_op_parallelism_threads(2)
    tf.config.threading.set_inter_op_parallelism_threads(1)

    logical_gpus = tf.config.list_logical_devices("GPU")

    print("Physical GPUs:", physical_gpus)
    print("Logical GPUs :", logical_gpus)

    return logical_gpus


available_gpus = configure_available_gpus()

In [ ]:
physical_gpus = tf.config.list_physical_devices("GPU")

print("Physical GPUs detected:", physical_gpus)

for gpu in physical_gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True
        )
        print(
            f"Memory growth enabled for: {gpu.name}"
        )

    except RuntimeError as error:
        # GPU is already initialized.
        # Do not crash the ESS execution because of this.
        print(
            f"GPU already initialized: {gpu.name}"
        )
        print(
            f"Skipping memory-growth configuration: {error}"
        )

logical_gpus = tf.config.list_logical_devices("GPU")

print(
    "Logical GPUs detected:",
    logical_gpus
)

## Data Generation

In [ ]:
# ============================================================
# GPU-ACCELERATED SYNTHETIC ORDINAL DATASET
# FOR ORDINAL LOGISTIC REGRESSION WITH
# ELLIPTICAL SLICE SAMPLING (ESS)
#
# Predictors:
#   Numerical   : X1, X2, X3
#   Categorical : Cat1, Cat2
#
# Target:
#   4 ordered categorical levels
#
# TensorFlow handles:
#   - Random-number generation
#   - Multivariate-normal generation
#   - Matrix operations
#   - Linear-predictor calculation
#   - Ordinal probability calculation
#   - Ordinal-target sampling
#
# pandas handles:
#   - Final DataFrame construction
#   - Ordered categorical target
#   - Dataset summaries
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os

# Reduce TensorFlow informational messages.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import tensorflow as tf


# ============================================================
# 2. BASIC CONFIGURATION
# ============================================================

SEED = 42

# Number of observations.
N = 3000

# Number of ordinal target categories.
K = 4

# TensorFlow floating-point data type.
TF_FLOAT = tf.float64

# Ordered target categories.
category_order = [
    "Category_1",
    "Category_2",
    "Category_3",
    "Category_4",
]

# Set the NumPy and TensorFlow global seeds.
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Allow TensorFlow to place unsupported GPU operations
# automatically on the CPU.
tf.config.set_soft_device_placement(True)


# ============================================================
# 3. CONFIGURE TENSORFLOW GPU
# ============================================================

physical_gpus = tf.config.list_physical_devices("GPU")

if len(physical_gpus) > 0:

    try:

        # Prevent TensorFlow from allocating all available
        # GPU memory immediately.
        for gpu in physical_gpus:
            tf.config.experimental.set_memory_growth(
                gpu,
                True,
            )

    except RuntimeError as error:

        # This can occur when the GPU has already been
        # initialized before memory growth is configured.
        print(
            "GPU memory-growth configuration warning:"
        )
        print(error)

    COMPUTE_DEVICE = "/GPU:0"

else:

    COMPUTE_DEVICE = "/CPU:0"


print("\n" + "=" * 70)
print("TENSORFLOW DEVICE CONFIGURATION")
print("=" * 70)

print(f"TensorFlow version          : {tf.__version__}")
print(f"Physical GPUs detected      : {len(physical_gpus)}")
print(f"Selected computation device : {COMPUTE_DEVICE}")

if len(physical_gpus) > 0:

    for gpu_number, gpu in enumerate(
        physical_gpus,
        start=1,
    ):
        print(
            f"GPU {gpu_number}                     : "
            f"{gpu.name}"
        )

else:

    print(
        "No TensorFlow-compatible GPU was detected."
    )

    print(
        "TensorFlow will execute the operations on the CPU."
    )


# ============================================================
# 4. DEFINE STATELESS RANDOM SEEDS
# ============================================================

# Different stateless seeds are used for the numerical
# predictors, Cat1, Cat2, and the target. This makes the
# results reproducible while keeping the random streams
# separate.

numerical_seed = tf.constant(
    [SEED, 1],
    dtype=tf.int32,
)

cat1_seed = tf.constant(
    [SEED, 2],
    dtype=tf.int32,
)

cat2_seed = tf.constant(
    [SEED, 3],
    dtype=tf.int32,
)

target_seed = tf.constant(
    [SEED, 4],
    dtype=tf.int32,
)


# ============================================================
# 5. DEFINE THE NUMERICAL COVARIANCE MATRIX
# ============================================================

with tf.device(COMPUTE_DEVICE):

    numerical_covariance_tensor = tf.constant(
        [
            [1.00, 0.15, 0.10],
            [0.15, 1.00, 0.15],
            [0.10, 0.15, 1.00],
        ],
        dtype=TF_FLOAT,
    )


# ============================================================
# 6. VALIDATE THE COVARIANCE MATRIX
# ============================================================

with tf.device(COMPUTE_DEVICE):

    covariance_eigenvalues_tensor = tf.linalg.eigvalsh(
        numerical_covariance_tensor
    )

    covariance_is_finite_tensor = tf.reduce_all(
        tf.math.is_finite(
            covariance_eigenvalues_tensor
        )
    )

    covariance_is_positive_definite_tensor = (
        tf.reduce_all(
            covariance_eigenvalues_tensor > 0.0
        )
    )


if not bool(covariance_is_finite_tensor.numpy()):
    raise ValueError(
        "The covariance matrix produced invalid eigenvalues."
    )

if not bool(
    covariance_is_positive_definite_tensor.numpy()
):
    raise ValueError(
        "The numerical covariance matrix must be "
        "positive definite."
    )


# ============================================================
# 7. GENERATE THREE CORRELATED NUMERICAL PREDICTORS
# ============================================================

with tf.device(COMPUTE_DEVICE):

    # Compute the lower-triangular Cholesky factor.
    covariance_cholesky_tensor = tf.linalg.cholesky(
        numerical_covariance_tensor
    )

    # Generate independent standard-normal values.
    standard_normal_tensor = tf.random.stateless_normal(
        shape=(N, 3),
        seed=numerical_seed,
        mean=0.0,
        stddev=1.0,
        dtype=TF_FLOAT,
    )

    # If:
    #
    # Z ~ Normal(0, I)
    #
    # and:
    #
    # covariance = L @ transpose(L)
    #
    # then:
    #
    # X = Z @ transpose(L)
    #
    # follows the required multivariate-normal distribution.
    X_numerical_tensor = tf.linalg.matmul(
        standard_normal_tensor,
        covariance_cholesky_tensor,
        transpose_b=True,
    )

    X1_tensor = X_numerical_tensor[:, 0]
    X2_tensor = X_numerical_tensor[:, 1]
    X3_tensor = X_numerical_tensor[:, 2]


# Validate numerical predictors.

with tf.device(COMPUTE_DEVICE):

    numerical_values_are_finite_tensor = tf.reduce_all(
        tf.math.is_finite(
            X_numerical_tensor
        )
    )


if not bool(
    numerical_values_are_finite_tensor.numpy()
):
    raise ValueError(
        "The generated numerical predictors contain "
        "invalid values."
    )


# ============================================================
# 8. GENERATE CAT1
# ============================================================

# Cat1 category encoding:
#
# 0 = A
# 1 = B
# 2 = C

with tf.device(COMPUTE_DEVICE):

    cat1_probabilities_tensor = tf.constant(
        [
            0.34,
            0.33,
            0.33,
        ],
        dtype=TF_FLOAT,
    )

    # tf.random.stateless_categorical expects logits.
    cat1_logits_tensor = tf.math.log(
        cat1_probabilities_tensor[
            tf.newaxis,
            :
        ]
    )

    # Output shape before squeezing:
    #
    # (1, N)
    Cat1_code_tensor = tf.random.stateless_categorical(
        logits=cat1_logits_tensor,
        num_samples=N,
        seed=cat1_seed,
        dtype=tf.int32,
    )

    # Convert shape from (1, N) to (N,).
    Cat1_code_tensor = tf.squeeze(
        Cat1_code_tensor,
        axis=0,
    )


# ============================================================
# 9. GENERATE CAT2
# ============================================================

# Cat2 category encoding:
#
# 0 = No
# 1 = Yes

with tf.device(COMPUTE_DEVICE):

    cat2_probabilities_tensor = tf.constant(
        [
            0.50,
            0.50,
        ],
        dtype=TF_FLOAT,
    )

    cat2_logits_tensor = tf.math.log(
        cat2_probabilities_tensor[
            tf.newaxis,
            :
        ]
    )

    # Output shape before squeezing:
    #
    # (1, N)
    Cat2_code_tensor = tf.random.stateless_categorical(
        logits=cat2_logits_tensor,
        num_samples=N,
        seed=cat2_seed,
        dtype=tf.int32,
    )

    # Convert shape from (1, N) to (N,).
    Cat2_code_tensor = tf.squeeze(
        Cat2_code_tensor,
        axis=0,
    )


# ============================================================
# 10. VALIDATE CATEGORICAL PREDICTOR SHAPES
# ============================================================

if tuple(Cat1_code_tensor.shape) != (N,):
    raise ValueError(
        "Cat1 has an incorrect shape. "
        f"Expected {(N,)}, "
        f"received {tuple(Cat1_code_tensor.shape)}."
    )

if tuple(Cat2_code_tensor.shape) != (N,):
    raise ValueError(
        "Cat2 has an incorrect shape. "
        f"Expected {(N,)}, "
        f"received {tuple(Cat2_code_tensor.shape)}."
    )


# ============================================================
# 11. DEFINE TRUE NUMERICAL EFFECTS
# ============================================================

with tf.device(COMPUTE_DEVICE):

    beta_numerical_true_tensor = tf.constant(
        [
             1.75,  # X1: positive effect
            -1.50,  # X2: negative effect
             1.25,  # X3: positive effect
        ],
        dtype=TF_FLOAT,
    )


# ============================================================
# 12. DEFINE TRUE CATEGORICAL EFFECTS
# ============================================================

# Cat1:
#
# A = reference category
# B = positive effect
# C = negative effect

cat1_effect = {
    "A": 0.00,
    "B": 1.20,
    "C": -1.20,
}

# Cat2:
#
# No  = reference category
# Yes = positive effect

cat2_effect = {
    "No": 0.00,
    "Yes": 1.00,
}


with tf.device(COMPUTE_DEVICE):

    # The position in this tensor corresponds to the
    # integer code used for Cat1.
    cat1_effect_tensor = tf.constant(
        [
            cat1_effect["A"],
            cat1_effect["B"],
            cat1_effect["C"],
        ],
        dtype=TF_FLOAT,
    )

    # The position in this tensor corresponds to the
    # integer code used for Cat2.
    cat2_effect_tensor = tf.constant(
        [
            cat2_effect["No"],
            cat2_effect["Yes"],
        ],
        dtype=TF_FLOAT,
    )


# ============================================================
# 13. CONSTRUCT THE LINEAR PREDICTOR
# ============================================================

with tf.device(COMPUTE_DEVICE):

    # Numerical contribution:
    #
    # eta_numerical = X @ beta
    eta_numerical_tensor = tf.linalg.matvec(
        X_numerical_tensor,
        beta_numerical_true_tensor,
    )

    # Select the correct Cat1 effect for every observation.
    eta_cat1_tensor = tf.gather(
        params=cat1_effect_tensor,
        indices=Cat1_code_tensor,
    )

    # Select the correct Cat2 effect for every observation.
    eta_cat2_tensor = tf.gather(
        params=cat2_effect_tensor,
        indices=Cat2_code_tensor,
    )

    # Complete systematic component.
    eta_tensor = (
        eta_numerical_tensor
        + eta_cat1_tensor
        + eta_cat2_tensor
    )

    eta_is_finite_tensor = tf.reduce_all(
        tf.math.is_finite(
            eta_tensor
        )
    )


if tuple(eta_tensor.shape) != (N,):
    raise ValueError(
        "The linear predictor has an incorrect shape. "
        f"Expected {(N,)}, "
        f"received {tuple(eta_tensor.shape)}."
    )

if not bool(eta_is_finite_tensor.numpy()):
    raise ValueError(
        "The linear predictor contains invalid values."
    )


# ============================================================
# 14. DEFINE THE TRUE ORDINAL CUTPOINTS
# ============================================================

# Four ordinal categories require three cutpoints.

with tf.device(COMPUTE_DEVICE):

    cutpoints_true_tensor = tf.constant(
        [
            -1.75,
             0.00,
             1.75,
        ],
        dtype=TF_FLOAT,
    )


# Validate the number of cutpoints.

number_of_cutpoints = int(
    tf.size(
        cutpoints_true_tensor
    ).numpy()
)

if number_of_cutpoints != K - 1:
    raise ValueError(
        f"{K} categories require exactly "
        f"{K - 1} cutpoints."
    )


# Validate cutpoint ordering.

with tf.device(COMPUTE_DEVICE):

    cutpoint_differences_tensor = (
        cutpoints_true_tensor[1:]
        - cutpoints_true_tensor[:-1]
    )

    cutpoints_are_ordered_tensor = tf.reduce_all(
        cutpoint_differences_tensor > 0.0
    )


if not bool(cutpoints_are_ordered_tensor.numpy()):
    raise ValueError(
        "The ordinal cutpoints must be strictly increasing."
    )


# ============================================================
# 15. CALCULATE CUMULATIVE ORDINAL PROBABILITIES
# ============================================================

# Proportional-odds ordinal logistic model:
#
# P(Y <= k | X) = sigmoid(alpha_k - eta)
#
# alpha_k = ordered cutpoint k
# eta     = X @ beta plus categorical effects

with tf.device(COMPUTE_DEVICE):

    cumulative_probabilities_tensor = tf.math.sigmoid(
        cutpoints_true_tensor[
            tf.newaxis,
            :
        ]
        - eta_tensor[
            :,
            tf.newaxis,
        ]
    )


# ============================================================
# 16. VALIDATE CUMULATIVE PROBABILITIES
# ============================================================

expected_cumulative_shape = (
    N,
    K - 1,
)

actual_cumulative_shape = tuple(
    cumulative_probabilities_tensor.shape
)

if actual_cumulative_shape != expected_cumulative_shape:
    raise ValueError(
        "The cumulative-probability matrix has an "
        "incorrect shape. "
        f"Expected {expected_cumulative_shape}, "
        f"received {actual_cumulative_shape}."
    )


with tf.device(COMPUTE_DEVICE):

    cumulative_probabilities_are_finite_tensor = (
        tf.reduce_all(
            tf.math.is_finite(
                cumulative_probabilities_tensor
            )
        )
    )

    cumulative_probabilities_in_range_tensor = (
        tf.reduce_all(
            tf.logical_and(
                cumulative_probabilities_tensor >= 0.0,
                cumulative_probabilities_tensor <= 1.0,
            )
        )
    )

    cumulative_probability_differences_tensor = (
        cumulative_probabilities_tensor[:, 1:]
        - cumulative_probabilities_tensor[:, :-1]
    )

    cumulative_probabilities_are_monotonic_tensor = (
        tf.reduce_all(
            cumulative_probability_differences_tensor
            >= -1e-12
        )
    )


if not bool(
    cumulative_probabilities_are_finite_tensor.numpy()
):
    raise ValueError(
        "The cumulative probabilities contain invalid values."
    )

if not bool(
    cumulative_probabilities_in_range_tensor.numpy()
):
    raise ValueError(
        "The cumulative probabilities are outside "
        "the interval [0, 1]."
    )

if not bool(
    cumulative_probabilities_are_monotonic_tensor.numpy()
):
    raise ValueError(
        "The cumulative probabilities are not "
        "monotonically increasing."
    )


# ============================================================
# 17. CONVERT CUMULATIVE PROBABILITIES TO
#     CATEGORY PROBABILITIES
# ============================================================

# For four categories:
#
# P(Y = 0) = P(Y <= 0)
#
# P(Y = 1) = P(Y <= 1) - P(Y <= 0)
#
# P(Y = 2) = P(Y <= 2) - P(Y <= 1)
#
# P(Y = 3) = 1 - P(Y <= 2)

with tf.device(COMPUTE_DEVICE):

    category_1_probability_tensor = (
        cumulative_probabilities_tensor[:, 0]
    )

    category_2_probability_tensor = (
        cumulative_probabilities_tensor[:, 1]
        - cumulative_probabilities_tensor[:, 0]
    )

    category_3_probability_tensor = (
        cumulative_probabilities_tensor[:, 2]
        - cumulative_probabilities_tensor[:, 1]
    )

    category_4_probability_tensor = (
        1.0
        - cumulative_probabilities_tensor[:, 2]
    )

    category_probabilities_tensor = tf.stack(
        [
            category_1_probability_tensor,
            category_2_probability_tensor,
            category_3_probability_tensor,
            category_4_probability_tensor,
        ],
        axis=1,
    )


# ============================================================
# 18. APPLY NUMERICAL SAFETY
# ============================================================

expected_category_probability_shape = (
    N,
    K,
)

actual_category_probability_shape = tuple(
    category_probabilities_tensor.shape
)

if (
    actual_category_probability_shape
    != expected_category_probability_shape
):
    raise ValueError(
        "The category-probability matrix has an "
        "incorrect shape. "
        f"Expected {expected_category_probability_shape}, "
        f"received {actual_category_probability_shape}."
    )


with tf.device(COMPUTE_DEVICE):

    # Clip small floating-point errors.
    category_probabilities_tensor = tf.clip_by_value(
        category_probabilities_tensor,
        clip_value_min=0.0,
        clip_value_max=1.0,
    )

    # Calculate the row totals.
    probability_row_sums_tensor = tf.reduce_sum(
        category_probabilities_tensor,
        axis=1,
        keepdims=True,
    )

    probability_sums_are_positive_tensor = tf.reduce_all(
        probability_row_sums_tensor > 0.0
    )


if not bool(
    probability_sums_are_positive_tensor.numpy()
):
    raise ValueError(
        "At least one observation has a zero total "
        "category probability."
    )


with tf.device(COMPUTE_DEVICE):

    # Normalize each probability row.
    category_probabilities_tensor = (
        category_probabilities_tensor
        / probability_row_sums_tensor
    )

    normalized_probability_sums_tensor = tf.reduce_sum(
        category_probabilities_tensor,
        axis=1,
    )

    probabilities_sum_to_one_tensor = tf.reduce_all(
        tf.abs(
            normalized_probability_sums_tensor - 1.0
        )
        <= 1e-10
    )

    category_probabilities_are_finite_tensor = (
        tf.reduce_all(
            tf.math.is_finite(
                category_probabilities_tensor
            )
        )
    )


if not bool(
    category_probabilities_are_finite_tensor.numpy()
):
    raise ValueError(
        "The category probabilities contain invalid values."
    )

if not bool(
    probabilities_sum_to_one_tensor.numpy()
):
    raise ValueError(
        "The category probabilities do not sum to one."
    )


# ============================================================
# 19. SAMPLE THE ORDINAL TARGET
# ============================================================

# The target must be sampled from the complete probability
# distribution.
#
# We do not use:
#
# tf.argmax(category_probabilities_tensor, axis=1)
#
# because argmax would make the response deterministic.
#
# tf.random.stateless_categorical expects logits, so the
# probabilities are converted to log probabilities.

with tf.device(COMPUTE_DEVICE):

    numerical_epsilon_tensor = tf.constant(
        1e-15,
        dtype=TF_FLOAT,
    )

    safe_category_probabilities_tensor = tf.maximum(
        category_probabilities_tensor,
        numerical_epsilon_tensor,
    )

    category_logits_tensor = tf.math.log(
        safe_category_probabilities_tensor
    )

    # One category is sampled independently for every row.
    #
    # Input shape:
    # (N, K)
    #
    # Output shape:
    # (N, 1)
    y_tensor = tf.random.stateless_categorical(
        logits=category_logits_tensor,
        num_samples=1,
        seed=target_seed,
        dtype=tf.int32,
    )

    # Convert shape from (N, 1) to (N,).
    y_tensor = tf.squeeze(
        y_tensor,
        axis=1,
    )


# ============================================================
# 20. VALIDATE THE GENERATED TARGET
# ============================================================

if tuple(y_tensor.shape) != (N,):
    raise ValueError(
        "The target has an incorrect shape. "
        f"Expected {(N,)}, "
        f"received {tuple(y_tensor.shape)}."
    )


with tf.device(COMPUTE_DEVICE):

    target_codes_are_valid_tensor = tf.reduce_all(
        tf.logical_and(
            y_tensor >= 0,
            y_tensor < K,
        )
    )

    category_counts_tensor = tf.math.bincount(
        y_tensor,
        minlength=K,
        maxlength=K,
        dtype=tf.int32,
    )

    all_categories_exist_tensor = tf.reduce_all(
        category_counts_tensor > 0
    )


if not bool(target_codes_are_valid_tensor.numpy()):
    raise ValueError(
        "The target contains invalid category codes."
    )

if not bool(all_categories_exist_tensor.numpy()):

    observed_categories = tf.where(
        category_counts_tensor > 0
    )[:, 0].numpy()

    raise ValueError(
        "Not all four ordinal categories were generated. "
        f"Observed categories: {observed_categories}"
    )


# ============================================================
# 21. TRANSFER RESULTS FROM TENSORFLOW TO NUMPY
# ============================================================

# pandas uses CPU-based NumPy arrays. Calling .numpy()
# transfers TensorFlow tensors from GPU memory to CPU memory.

X_numerical = X_numerical_tensor.numpy()

X1 = X_numerical[:, 0]
X2 = X_numerical[:, 1]
X3 = X_numerical[:, 2]

Cat1_code = Cat1_code_tensor.numpy()
Cat2_code = Cat2_code_tensor.numpy()

y = y_tensor.numpy().astype(
    np.int32
)

beta_numerical_true = (
    beta_numerical_true_tensor.numpy()
)

cutpoints_true = (
    cutpoints_true_tensor.numpy()
)

cumulative_probabilities = (
    cumulative_probabilities_tensor.numpy()
)

category_probabilities = (
    category_probabilities_tensor.numpy()
)


# ============================================================
# 22. CONVERT CATEGORY CODES TO LABELS
# ============================================================

cat1_labels = np.array(
    [
        "A",
        "B",
        "C",
    ],
    dtype=object,
)

cat2_labels = np.array(
    [
        "No",
        "Yes",
    ],
    dtype=object,
)

target_labels = np.array(
    category_order,
    dtype=object,
)

Cat1 = cat1_labels[Cat1_code]
Cat2 = cat2_labels[Cat2_code]
target = target_labels[y]


# ============================================================
# 23. CREATE THE FINAL DATAFRAME
# ============================================================

df = pd.DataFrame(
    {
        "X1": X1,
        "X2": X2,
        "X3": X3,
        "Cat1": Cat1,
        "Cat2": Cat2,
        "target_code": y,
        "target": target,
    }
)


# ============================================================
# 24. PRESERVE CATEGORICAL PREDICTOR STRUCTURES
# ============================================================

df["Cat1"] = pd.Categorical(
    df["Cat1"],
    categories=[
        "A",
        "B",
        "C",
    ],
    ordered=False,
)

df["Cat2"] = pd.Categorical(
    df["Cat2"],
    categories=[
        "No",
        "Yes",
    ],
    ordered=False,
)


# ============================================================
# 25. PRESERVE THE ORDINAL TARGET STRUCTURE
# ============================================================

df["target"] = pd.Categorical(
    df["target"],
    categories=category_order,
    ordered=True,
)


# ============================================================
# 26. VALIDATE THE FINAL DATAFRAME
# ============================================================

expected_columns = [
    "X1",
    "X2",
    "X3",
    "Cat1",
    "Cat2",
    "target_code",
    "target",
]

if df.shape != (N, 7):
    raise ValueError(
        "The final DataFrame has an incorrect shape. "
        f"Expected {(N, 7)}, "
        f"received {df.shape}."
    )

if df.columns.tolist() != expected_columns:
    raise ValueError(
        "The final DataFrame contains unexpected columns. "
        f"Expected {expected_columns}, "
        f"received {df.columns.tolist()}."
    )

if df.isna().any().any():
    raise ValueError(
        "The final DataFrame contains missing values."
    )

if not df["target"].cat.ordered:
    raise ValueError(
        "The target variable is not stored as an "
        "ordered categorical variable."
    )


# ============================================================
# 27. DISPLAY THE FIRST 10 OBSERVATIONS
# ============================================================

print("\n" + "=" * 70)
print("FIRST 10 OBSERVATIONS")
print("=" * 70)

print(
    df.head(10).to_string(
        index=False
    )
)


# ============================================================
# 28. DISPLAY TARGET DISTRIBUTION
# ============================================================

target_distribution = (
    df["target"]
    .value_counts(sort=False)
    .reindex(
        category_order,
        fill_value=0,
    )
)

target_percentage = (
    target_distribution
    / N
    * 100.0
)


print("\n" + "=" * 70)
print("ORDINAL TARGET DISTRIBUTION")
print("=" * 70)

for category in category_order:

    count = int(
        target_distribution.loc[category]
    )

    percentage = float(
        target_percentage.loc[category]
    )

    print(
        f"{category:12s}: "
        f"{count:4d} "
        f"({percentage:6.2f}%)"
    )


# ============================================================
# 29. DISPLAY CAT1 DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CAT1 DISTRIBUTION")
print("=" * 70)

print(
    df["Cat1"]
    .value_counts(sort=False)
    .sort_index()
)


# ============================================================
# 30. DISPLAY CAT2 DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CAT2 DISTRIBUTION")
print("=" * 70)

print(
    df["Cat2"]
    .value_counts(sort=False)
    .sort_index()
)


# ============================================================
# DISPLAY TRUE DATA-GENERATING PARAMETERS
# ============================================================

print("\n" + "=" * 70)
print("TRUE DATA-GENERATING PARAMETERS")
print("=" * 70)

print("\nNumerical effects:")

print(
    "X1      : {:.2f}".format(
        float(beta_numerical_true[0])
    )
)

print(
    "X2      : {:.2f}".format(
        float(beta_numerical_true[1])
    )
)

print(
    "X3      : {:.2f}".format(
        float(beta_numerical_true[2])
    )
)


print("\nCategorical effects:")

print(
    "Cat1_A  : {:.2f}  [Reference]".format(
        float(cat1_effect["A"])
    )
)

print(
    "Cat1_B  : {:.2f}".format(
        float(cat1_effect["B"])
    )
)

print(
    "Cat1_C  : {:.2f}".format(
        float(cat1_effect["C"])
    )
)

print(
    "Cat2_No : {:.2f}  [Reference]".format(
        float(cat2_effect["No"])
    )
)

print(
    "Cat2_Yes: {:.2f}".format(
        float(cat2_effect["Yes"])
    )
)


print("\nTrue cutpoints:")

for index, cutpoint in enumerate(
    cutpoints_true,
    start=1
):
    print(
        "Cutpoint {}: {:.2f}".format(
            index,
            float(cutpoint)
        )
    )


# ============================================================
# 32. DISPLAY TENSORFLOW COMPUTATION INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("TENSORFLOW COMPUTATION INFORMATION")
print("=" * 70)

print(
    f"Requested device          : "
    f"{COMPUTE_DEVICE}"
)

print(
    f"Numerical tensor device   : "
    f"{X_numerical_tensor.device}"
)

print(
    f"Linear predictor device   : "
    f"{eta_tensor.device}"
)

print(
    f"Probability tensor device : "
    f"{category_probabilities_tensor.device}"
)

print(
    f"Target tensor device      : "
    f"{y_tensor.device}"
)

print(
    f"TensorFlow numerical type : "
    f"{X_numerical_tensor.dtype.name}"
)


# ============================================================
# 33. DISPLAY FINAL DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    f"Number of observations : "
    f"{df.shape[0]}"
)

print(
    "Number of predictors   : 5"
)

print(
    "Numerical predictors   : X1, X2, X3"
)

print(
    "Categorical predictors : Cat1, Cat2"
)

print(
    "Ordinal target         : target"
)

print(
    "Target code            : target_code"
)

print(
    f"Number of categories   : {K}"
)

print(
    f"Target ordered         : "
    f"{df['target'].cat.ordered}"
)

print(
    f"Dataset shape          : {df.shape}"
)


print("\nFinal column names:")

print(
    df.columns.tolist()
)


print("\nFirst 10 rows:")

print(
    df.head(10).to_string(
        index=False
    )
)

In [ ]:
df.to_csv('synthetic_sample_ess.csv',index = False)

In [ ]:
# ============================================================
# 0. IMPORT REQUIRED LIBRARIES
# GPU / TENSORFLOW VERSION
# ============================================================

import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow_probability as tfp
import arviz as az

import scipy.special

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss,
    mean_absolute_error,
    cohen_kappa_score
)

# ============================================================
# TENSORFLOW PROBABILITY SHORTHANDS
# ============================================================

tfd = tfp.distributions


# ============================================================
# GPU CONFIGURATION
# ============================================================

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print(f"GPU(s) detected: {len(gpus)}")

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass

    print("TensorFlow GPU devices:")
    for gpu in gpus:
        print(gpu)

else:
    print("No GPU detected. TensorFlow will run on CPU.")


# ============================================================
# DEFAULT TENSOR DATA TYPES
# ============================================================

TF_FLOAT = tf.float32
TF_INT = tf.int32

print("\nTensorFlow version:", tf.__version__)
print("TensorFlow Probability version:", tfp.__version__)
print("Default float tensor dtype:", TF_FLOAT)
print("Default integer tensor dtype:", TF_INT)


# ============================================================
# HELPER FUNCTIONS FOR TENSOR CONVERSION
# ============================================================

def to_float_tensor(x):
    """
    Convert NumPy array / pandas object / Python object
    to TensorFlow float32 tensor.
    """
    return tf.convert_to_tensor(x, dtype=TF_FLOAT)


def to_int_tensor(x):
    """
    Convert NumPy array / pandas object / Python object
    to TensorFlow int32 tensor.
    """
    return tf.convert_to_tensor(x, dtype=TF_INT)

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

# Random seed for reproducibility.
SEED = 45

# Proportion of observations assigned to the test set.
TEST_SIZE = 0.20

# Number of independent MCMC chains.
N_CHAINS = 4

# Number of retained posterior samples per chain.
N_SAMPLES = 3000

# Number of initial burn-in or warm-up iterations.
N_BURNIN = 1500

# Float64 for MCMC and log-probability calculations.
DTYPE = tf.float64


# ============================================================
# RANDOM SEEDS
# ============================================================

# NumPy seed.
# Retained because NumPy / scikit-learn may still be used
# during preprocessing and train/test preparation.
np.random.seed(SEED)

# TensorFlow seed used by TensorFlow random operations.
tf.random.set_seed(SEED)


# ============================================================
# TENSOR CONSTANTS
# ============================================================

# TensorFlow scalar constants used frequently in MCMC calculations.
TF_ZERO = tf.constant(0.0, dtype=DTYPE)
TF_ONE = tf.constant(1.0, dtype=DTYPE)
TF_TWO = tf.constant(2.0, dtype=DTYPE)

TF_PI = tf.constant(np.pi, dtype=DTYPE)

# Seed as TensorFlow integer tensor when required.
TF_SEED = tf.constant(SEED, dtype=tf.int32)


# ============================================================
# CHECK COMPUTE DEVICE
# ============================================================

GPU_AVAILABLE = len(tf.config.list_physical_devices("GPU")) > 0

if GPU_AVAILABLE:
    print("GPU processing available.")
    print("GPU devices:", tf.config.list_physical_devices("GPU"))
else:
    print("No GPU detected. TensorFlow will use CPU.")

print("TensorFlow dtype:", DTYPE)
print("Random seed:", SEED)
print("MCMC chains:", N_CHAINS)
print("Posterior samples per chain:", N_SAMPLES)
print("Burn-in iterations:", N_BURNIN)

In [ ]:
df.columns

In [ ]:
# ============================================================
# 2. LOAD AND PREPARE THE DATASET
#    CPU preprocessing + TensorFlow GPU model tensors
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import OneHotEncoder


# ============================================================
# 2.0 TENSORFLOW DEVICE CONFIGURATION
# ============================================================

# Allow TensorFlow to place operations on the CPU automatically
# if a particular GPU kernel is unavailable.
tf.config.set_soft_device_placement(True)

physical_gpus = tf.config.list_physical_devices("GPU")

if len(physical_gpus) > 0:

    # Memory growth may already have been configured in the
    # previous section. Therefore, only try to configure it
    # if possible.
    try:

        for gpu in physical_gpus:

            tf.config.experimental.set_memory_growth(
                gpu,
                True
            )

    except RuntimeError:

        # Ignore this if TensorFlow has already initialized
        # the GPU earlier in the notebook.
        pass

    COMPUTE_DEVICE = "/GPU:0"

else:

    COMPUTE_DEVICE = "/CPU:0"


print("\n" + "=" * 60)
print("TENSORFLOW DEVICE")
print("=" * 60)

print(
    "TensorFlow version : {}".format(
        tf.__version__
    )
)

print(
    "GPUs detected      : {}".format(
        len(physical_gpus)
    )
)

print(
    "Compute device     : {}".format(
        COMPUTE_DEVICE
    )
)


# ============================================================
# 2.1 DEFINE TARGET COLUMN
# ============================================================

target_column = "target"


# ============================================================
# 2.2 DEFINE CATEGORICAL PREDICTORS
# ============================================================

categorical_columns = [
    "Cat1",
    "Cat2"
]


# ============================================================
# 2.3 VERIFY REQUIRED COLUMNS EXIST
# ============================================================

basic_required_columns = (
    categorical_columns
    + [
        target_column
    ]
)

missing_required_columns = [
    column
    for column in basic_required_columns
    if column not in df.columns
]

if len(missing_required_columns) > 0:

    raise ValueError(
        "The following required columns are missing "
        "from df: {}".format(
            missing_required_columns
        )
    )


# ============================================================
# 2.4 DEFINE ALL PREDICTORS
# ============================================================

# Exclude:
#
# 1. target
# 2. target_code
#
# target_code must not be used because it directly encodes
# the response variable and would cause target leakage.

columns_to_exclude = [
    target_column,
    "target_code"
]

predictor_columns_raw = [
    column
    for column in df.columns
    if column not in columns_to_exclude
]


if len(predictor_columns_raw) == 0:

    raise ValueError(
        "No predictor columns were found."
    )


# ============================================================
# 2.5 IDENTIFY NUMERICAL PREDICTORS
# ============================================================

numerical_columns = [
    column
    for column in predictor_columns_raw
    if column not in categorical_columns
]


if len(numerical_columns) == 0:

    raise ValueError(
        "No numerical predictors were identified."
    )


print("\nRAW PREDICTORS")
print("=" * 60)

print(
    "Numerical predictors:",
    numerical_columns
)

print(
    "Categorical predictors:",
    categorical_columns
)


# ============================================================
# 2.6 REMOVE MISSING OBSERVATIONS
# ============================================================

required_columns = (
    numerical_columns
    + categorical_columns
    + [target_column]
)


df_model = (
    df[required_columns]
    .dropna()
    .copy()
)


df_model.reset_index(
    drop=True,
    inplace=True
)


if len(df_model) == 0:

    raise ValueError(
        "No observations remain after removing missing values."
    )


print(
    "\nObservations after removing missing values: {}".format(
        len(df_model)
    )
)


# ============================================================
# 2.7 ONE-HOT ENCODE CATEGORICAL PREDICTORS
# ============================================================

# drop="first" produces reference-category coding.
#
# Cat1:
#
# A -> reference
# B -> Cat1_B
# C -> Cat1_C
#
# Cat2:
#
# No  -> reference
# Yes -> Cat2_Yes


encoder = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore",
    dtype=np.float64
)


X_categorical_cpu = encoder.fit_transform(
    df_model[categorical_columns]
)


X_categorical_cpu = np.asarray(
    X_categorical_cpu,
    dtype=np.float64
)


categorical_encoded_columns = (
    encoder
    .get_feature_names_out(
        categorical_columns
    )
    .tolist()
)


print("\nONE-HOT ENCODED COLUMNS")
print("=" * 60)

for column in categorical_encoded_columns:

    print(column)


# ============================================================
# 2.8 EXTRACT NUMERICAL PREDICTORS
# ============================================================

X_numerical_cpu = (
    df_model[numerical_columns]
    .to_numpy(
        dtype=np.float64
    )
)


# ============================================================
# 2.9 COMBINE NUMERICAL + CATEGORICAL PREDICTORS
# ============================================================

# Final CPU model matrix:
#
# [numerical predictors | one-hot categorical predictors]


X_cpu = np.column_stack(
    [
        X_numerical_cpu,
        X_categorical_cpu
    ]
).astype(
    np.float64,
    copy=False
)


predictor_columns = (
    numerical_columns
    + categorical_encoded_columns
)


if X_cpu.shape[1] != len(predictor_columns):

    raise ValueError(
        "The number of predictor names does not match "
        "the number of columns in X."
    )


print("\nFINAL MODEL PREDICTORS")
print("=" * 60)

for j, column in enumerate(
    predictor_columns,
    start=1
):

    print(
        "{}. {}".format(
            j,
            column
        )
    )


# ============================================================
# 2.10 EXTRACT ORIGINAL ORDINAL TARGET
# ============================================================

# Convert to object/string-like NumPy representation.
#
# This works whether target was previously stored as a
# pandas ordered categorical or as a standard object column.

y_original = (
    df_model[target_column]
    .astype(object)
    .to_numpy()
)


# ============================================================
# 2.11 DEFINE THE CORRECT ORDINAL CATEGORY ORDER
# ============================================================

category_order = np.array(
    [
        "Category_1",
        "Category_2",
        "Category_3",
        "Category_4"
    ],
    dtype=object
)


# ============================================================
# 2.12 CREATE ORDINAL CATEGORY MAP
# ============================================================

category_map = {
    category: index
    for index, category in enumerate(
        category_order
    )
}


# ============================================================
# 2.13 CHECK FOR UNKNOWN TARGET CATEGORIES
# ============================================================

observed_target_categories = set(
    np.unique(
        y_original
    ).tolist()
)

expected_target_categories = set(
    category_order.tolist()
)


unknown_categories = (
    observed_target_categories
    - expected_target_categories
)


if len(unknown_categories) > 0:

    raise ValueError(
        "The target contains categories that are not "
        "present in category_order: {}".format(
            unknown_categories
        )
    )


# ============================================================
# 2.14 CONVERT ORDINAL TARGET TO INTEGER LABELS
# ============================================================

# Category_1 -> 0
# Category_2 -> 1
# Category_3 -> 2
# Category_4 -> 3


y_cpu = np.array(
    [
        category_map[value]
        for value in y_original
    ],
    dtype=np.int32
)


K = len(
    category_order
)


# ============================================================
# 2.15 BASIC CPU VALIDATION
# ============================================================

if K < 2:

    raise ValueError(
        "At least two ordinal outcome categories are required."
    )


if X_cpu.ndim != 2:

    raise ValueError(
        "X must be a two-dimensional predictor matrix."
    )


if y_cpu.ndim != 1:

    raise ValueError(
        "y must be a one-dimensional target vector."
    )


if X_cpu.shape[0] != y_cpu.shape[0]:

    raise ValueError(
        "X and y must contain the same number "
        "of observations."
    )


if not np.all(
    np.isfinite(
        X_cpu
    )
):

    raise ValueError(
        "X contains missing or infinite values."
    )


if np.any(y_cpu < 0) or np.any(y_cpu >= K):

    raise ValueError(
        "The ordinal target contains invalid integer codes."
    )


# ============================================================
# 2.16 CHECK CATEGORY COUNTS
# ============================================================

category_counts = np.bincount(
    y_cpu,
    minlength=K
)


if len(category_counts) != K:

    raise ValueError(
        "Target category count calculation failed."
    )


if np.any(
    category_counts < 2
):

    raise ValueError(
        "Every ordinal category must contain at least "
        "two observations to use a stratified "
        "train-test split."
    )


# ============================================================
# 2.17 TRANSFER MODEL DATA TO TENSORFLOW DEVICE
# ============================================================

# The CPU preprocessing is now complete.
#
# Convert only the final modeling arrays into TensorFlow
# tensors. If COMPUTE_DEVICE is /GPU:0, these tensors are
# placed on the GPU.


with tf.device(COMPUTE_DEVICE):

    X_tensor = tf.convert_to_tensor(
        X_cpu,
        dtype=tf.float64
    )

    y_tensor = tf.convert_to_tensor(
        y_cpu,
        dtype=tf.int32
    )


# ============================================================
# 2.18 TENSORFLOW VALIDATION
# ============================================================

with tf.device(COMPUTE_DEVICE):

    X_is_finite_tensor = tf.reduce_all(
        tf.math.is_finite(
            X_tensor
        )
    )

    y_min_tensor = tf.reduce_min(
        y_tensor
    )

    y_max_tensor = tf.reduce_max(
        y_tensor
    )


if not bool(
    X_is_finite_tensor.numpy()
):

    raise ValueError(
        "The TensorFlow predictor matrix contains "
        "non-finite values."
    )


if int(
    y_min_tensor.numpy()
) < 0:

    raise ValueError(
        "The TensorFlow target contains negative "
        "category codes."
    )


if int(
    y_max_tensor.numpy()
) >= K:

    raise ValueError(
        "The TensorFlow target contains category codes "
        "outside the expected ordinal range."
    )


# ============================================================
# 2.19 CREATE STANDARD X AND y VARIABLES
# ============================================================

# From this point onward, use:
#
# X -> TensorFlow tensor
# y -> TensorFlow tensor
#
# in the GPU-based ordinal ESS implementation.
#
# CPU copies remain available as:
#
# X_cpu
# y_cpu


X = X_tensor
y = y_tensor


# ============================================================
# 2.20 DISPLAY FINAL DATA INFORMATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL DATASET FOR ORDINAL ESS")
print("=" * 60)


print(
    "Number of observations : {}".format(
        X.shape[0]
    )
)

print(
    "Number of predictors   : {}".format(
        X.shape[1]
    )
)

print(
    "Number of categories   : {}".format(
        K
    )
)

print(
    "X shape                : {}".format(
        tuple(X.shape)
    )
)

print(
    "y shape                : {}".format(
        tuple(y.shape)
    )
)

print(
    "X dtype                : {}".format(
        X.dtype.name
    )
)

print(
    "y dtype                : {}".format(
        y.dtype.name
    )
)


# ============================================================
# 2.21 DISPLAY TENSOR DEVICES
# ============================================================

print("\nTENSORFLOW DEVICE INFORMATION")
print("=" * 60)

print(
    "Requested device : {}".format(
        COMPUTE_DEVICE
    )
)

print(
    "X device         : {}".format(
        X.device
    )
)

print(
    "y device         : {}".format(
        y.device
    )
)


# ============================================================
# 2.22 DISPLAY PREDICTORS
# ============================================================

print("\nPREDICTORS")
print("=" * 60)


for j, predictor in enumerate(
    predictor_columns,
    start=1
):

    print(
        "{:2d}. {}".format(
            j,
            predictor
        )
    )


# ============================================================
# 2.23 DISPLAY ORDINAL TARGET MAPPING
# ============================================================

print("\nORDINAL TARGET MAPPING")
print("=" * 60)


for category, code in category_map.items():

    print(
        "{:15s} -> {}".format(
            str(category),
            code
        )
    )


# ============================================================
# 2.24 DISPLAY TARGET DISTRIBUTION
# ============================================================

print("DISTRIBUTION")
print("=" * 60)


for code, category in enumerate(
    category_order
):

    count = int(
        category_counts[code]
    )

    percentage = (
        count
        / len(y_cpu)
        * 100.0
    )

    print(
        "{:15s}: {:4d} ({:5.2f}%)".format(
            str(category),
            count,
            percentage
        )
    )


# ============================================================
# 2.25 FINAL VALIDATION MESSAGE
# ============================================================

print("\n" + "=" * 60)
print("DATA PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "CPU predictor matrix : X_cpu"
)

print(
    "CPU target vector    : y_cpu"
)

print(
    "TensorFlow matrix    : X"
)

print(
    "TensorFlow target    : y"
)

print(
    "TensorFlow device    : {}".format(
        COMPUTE_DEVICE
    )
)

In [ ]:
# ============================================================
# 3. TRAIN-TEST SPLIT
# TENSORFLOW / GPU VERSION
# ============================================================

# Convert input data to TensorFlow tensors.
X_tensor = tf.convert_to_tensor(X, dtype=DTYPE)
y_tensor = tf.convert_to_tensor(y, dtype=tf.int32)

# ------------------------------------------------------------
# Stratified train-test split
# ------------------------------------------------------------

# Get unique classes.
classes = tf.unique(y_tensor).y

train_indices_list = []
test_indices_list = []

for cls in classes:

    # Find indices belonging to the current class.
    cls_indices = tf.where(
        tf.equal(y_tensor, cls)
    )[:, 0]

    # Shuffle class-specific indices reproducibly.
    cls_indices = tf.random.shuffle(
        cls_indices,
        seed=SEED
    )

    # Number of observations in this class.
    n_cls = tf.shape(cls_indices)[0]

    # Number of test observations.
    n_test = tf.cast(
        tf.round(
            tf.cast(n_cls, DTYPE)
            * tf.cast(TEST_SIZE, DTYPE)
        ),
        tf.int32
    )

    # Split indices.
    test_indices_cls = cls_indices[:n_test]
    train_indices_cls = cls_indices[n_test:]

    train_indices_list.append(train_indices_cls)
    test_indices_list.append(test_indices_cls)


# Combine indices from all classes.
train_indices = tf.concat(
    train_indices_list,
    axis=0
)

test_indices = tf.concat(
    test_indices_list,
    axis=0
)


# ------------------------------------------------------------
# Shuffle final train/test indices
# ------------------------------------------------------------

train_indices = tf.random.shuffle(
    train_indices,
    seed=SEED
)

test_indices = tf.random.shuffle(
    test_indices,
    seed=SEED + 1
)


# ------------------------------------------------------------
# Create train/test tensors
# ------------------------------------------------------------

X_train = tf.gather(
    X_tensor,
    train_indices
)

X_test = tf.gather(
    X_tensor,
    test_indices
)

y_train = tf.gather(
    y_tensor,
    train_indices
)

y_test = tf.gather(
    y_tensor,
    test_indices
)


# ------------------------------------------------------------
# Information
# ------------------------------------------------------------

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

print("\nX_train dtype:", X_train.dtype)
print("X_test dtype :", X_test.dtype)
print("y_train dtype:", y_train.dtype)
print("y_test dtype :", y_test.dtype)

In [ ]:
# ============================================================
# 4. STANDARDIZE ONLY THE NUMERICAL PREDICTORS
# TENSORFLOW / GPU VERSION
# ============================================================

# ------------------------------------------------------------
# The design matrix contains:
#
# Column 0 : X1
# Column 1 : X2
# Column 2 : X3
# Column 3 : Cat1_B
# Column 4 : Cat1_C
# Column 5 : Cat2_Yes
#
# Only X1, X2 and X3 should be standardized.
# Dummy-coded categorical predictors remain 0/1.
# ------------------------------------------------------------


# ------------------------------------------------------------
# COLUMN INDICES
# ------------------------------------------------------------

# Indices of numerical predictors.
numerical_indices = tf.constant(
    [0, 1, 2],
    dtype=tf.int32
)

# Indices of categorical dummy predictors.
categorical_indices = tf.constant(
    [3, 4, 5],
    dtype=tf.int32
)


# ------------------------------------------------------------
# ENSURE CORRECT TENSOR DTYPE
# ------------------------------------------------------------

X_train = tf.cast(
    X_train,
    dtype=DTYPE
)

X_test = tf.cast(
    X_test,
    dtype=DTYPE
)


# ------------------------------------------------------------
# EXTRACT NUMERICAL PREDICTORS
# ------------------------------------------------------------

X_train_numerical = tf.gather(
    X_train,
    numerical_indices,
    axis=1
)

X_test_numerical = tf.gather(
    X_test,
    numerical_indices,
    axis=1
)


# ------------------------------------------------------------
# EXTRACT CATEGORICAL PREDICTORS
# ------------------------------------------------------------

X_train_categorical = tf.gather(
    X_train,
    categorical_indices,
    axis=1
)

X_test_categorical = tf.gather(
    X_test,
    categorical_indices,
    axis=1
)


# ------------------------------------------------------------
# COMPUTE TRAINING MEAN AND STANDARD DEVIATION
# ------------------------------------------------------------
#
# Equivalent idea to:
#
# scaler.fit(X_train[:, numerical_indices])
#
# IMPORTANT:
# Statistics are calculated ONLY from the training data.
# ------------------------------------------------------------

train_mean = tf.reduce_mean(
    X_train_numerical,
    axis=0
)

train_std = tf.math.reduce_std(
    X_train_numerical,
    axis=0
)


# ------------------------------------------------------------
# PROTECT AGAINST ZERO STANDARD DEVIATION
# ------------------------------------------------------------
#
# If a numerical predictor is constant in the training set,
# its standard deviation will be zero.
#
# Replace zero standard deviations with 1.0 to prevent
# division by zero.
# ------------------------------------------------------------

train_std = tf.where(
    train_std > tf.constant(0.0, dtype=DTYPE),
    train_std,
    tf.ones_like(train_std)
)


# ------------------------------------------------------------
# STANDARDIZE TRAINING NUMERICAL PREDICTORS
# ------------------------------------------------------------

X_train_numerical_scaled = (
    X_train_numerical - train_mean
) / train_std


# ------------------------------------------------------------
# STANDARDIZE TEST NUMERICAL PREDICTORS
# ------------------------------------------------------------
#
# Use EXACTLY the mean and standard deviation calculated
# from the training data.
#
# No information from X_test is used to calculate the
# scaling parameters.
# ------------------------------------------------------------

X_test_numerical_scaled = (
    X_test_numerical - train_mean
) / train_std


# ------------------------------------------------------------
# RECONSTRUCT THE DESIGN MATRICES
# ------------------------------------------------------------
#
# Original column order:
#
# X1, X2, X3, Cat1_B, Cat1_C, Cat2_Yes
#
# Therefore, standardized numerical predictors are concatenated
# with the unchanged categorical dummy predictors.
# ------------------------------------------------------------

X_train = tf.concat(
    [
        X_train_numerical_scaled,
        X_train_categorical
    ],
    axis=1
)

X_test = tf.concat(
    [
        X_test_numerical_scaled,
        X_test_categorical
    ],
    axis=1
)


# ------------------------------------------------------------
# NUMBER OF TRAINING OBSERVATIONS AND PREDICTORS
# ------------------------------------------------------------

n_train = tf.shape(X_train)[0]
p = tf.shape(X_train)[1]


# ------------------------------------------------------------
# CHECK THE RESULT
# ------------------------------------------------------------

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nX_train dtype:", X_train.dtype)
print("X_test dtype :", X_test.dtype)

print("\nNumerical columns standardized:")
print("X1, X2, X3")

print("\nCategorical columns NOT standardized:")
print("Cat1_B, Cat1_C, Cat2_Yes")


# ------------------------------------------------------------
# CHECK TRAINING MEANS AND STANDARD DEVIATIONS
# ------------------------------------------------------------

print("\nMeans of standardized numerical predictors:")

tf.print(
    tf.reduce_mean(
        X_train[:, :3],
        axis=0
    )
)

print("\nStandard deviations of standardized numerical predictors:")

tf.print(
    tf.math.reduce_std(
        X_train[:, :3],
        axis=0
    )
)


# ------------------------------------------------------------
# UNIQUE VALUES OF CATEGORICAL PREDICTORS
# ------------------------------------------------------------

print("\nUnique values of categorical predictors in training data:")

for j in [3, 4, 5]:

    unique_values, _ = tf.unique(
        X_train[:, j]
    )

    tf.print(
        "Column",
        j,
        ":",
        unique_values
    )

In [ ]:
# ============================================================
# 5. CONVERT TRAINING DATA TO TENSORFLOW TENSORS
# ============================================================

X_train_tf = tf.constant(
    X_train,
    dtype=DTYPE
)

y_train_tf = tf.constant(
    y_train,
    dtype=tf.int32
)

In [ ]:
# ============================================================
# 6. PRIOR SPECIFICATION
# ============================================================

# The full parameter vector is:
#
# theta = [
#     beta_1, ..., beta_p,
#     gamma_1, ..., gamma_(K-1)
# ]
#
# beta contains the regression coefficients.
# gamma contains unconstrained parameters used to construct ordered cutpoints.
#
# The prior model is:
#
# theta ~ Normal(prior_mean, prior_cov)
#
# Elliptical Slice Sampling is performed using whitened parameters:
#
# theta = prior_mean + L @ z
#
# where:
#
# z ~ Normal(0, I)
# L @ L.T = prior_cov

# Total number of unconstrained model parameters.
d = p + K - 1

# Zero-centred prior.
prior_mean = np.zeros(d, dtype=np.float64)

# Prior standard deviations:
# 2.5 for regression coefficients and
# 1.5 for unconstrained cutpoint parameters.
prior_sd = np.concatenate(
    [
        np.full(p, 2.5),
        np.full(K - 1, 1.5)
    ]
)

# Diagonal prior covariance matrix.
prior_cov = np.diag(prior_sd ** 2)

# Cholesky factor of the prior covariance matrix.
L = np.linalg.cholesky(prior_cov)

# Convert prior quantities to TensorFlow tensors.
prior_mean_tf = tf.constant(
    prior_mean,
    dtype=DTYPE
)

L_tf = tf.constant(
    L,
    dtype=DTYPE
)

In [ ]:
# ============================================================
# 7. PARAMETER TRANSFORMATION
# ============================================================

def construct_ordered_cutpoints_tf(gamma):
    """
    Convert unconstrained TensorFlow parameters into ordered cutpoints.

    Parameters
    ----------
    gamma : Tensor
        Shape [..., K - 1].

    Returns
    -------
    alpha : Tensor
        Strictly ordered cutpoints with shape [..., K - 1].
    """

    # The first cutpoint remains unconstrained.
    first_cutpoint = gamma[..., :1]

    if K == 2:
        # A binary outcome requires only one cutpoint.
        return first_cutpoint

    # Softplus creates positive increments and is numerically safer
    # than exp(gamma), which can overflow for large gamma values.
    positive_increments = (
        tf.nn.softplus(gamma[..., 1:])
        + tf.cast(1e-8, DTYPE)
    )

    # Add cumulative positive increments to the first cutpoint.
    remaining_cutpoints = (
        first_cutpoint
        + tf.cumsum(
            positive_increments,
            axis=-1
        )
    )

    return tf.concat(
        [
            first_cutpoint,
            remaining_cutpoints
        ],
        axis=-1
    )


def transform_parameters(z):
    """
    Transform one or more whitened parameter vectors into beta and
    ordered cutpoints.

    Parameters
    ----------
    z : Tensor
        Whitened parameter vector with final dimension d.

    Returns
    -------
    beta : Tensor
        Regression coefficient parameters.

    alpha : Tensor
        Ordered cutpoints.
    """

    # For row-vector samples:
    #
    # theta = prior_mean + z @ L.T
    theta = (
        prior_mean_tf
        + tf.linalg.matmul(
            z,
            L_tf,
            transpose_b=True
        )
    )

    # Extract regression coefficients.
    beta = theta[..., :p]

    # Extract unconstrained cutpoint parameters.
    gamma = theta[..., p:]

    # Transform gamma into ordered cutpoints.
    alpha = construct_ordered_cutpoints_tf(gamma)

    return beta, alpha

In [ ]:
# ============================================================
# 8. LOG-LIKELIHOOD
# ============================================================

def logdiffexp(a, b):
    """
    Compute log(exp(a) - exp(b)) in a numerically stable manner.

    This function assumes that a >= b.
    """

    # Clipping prevents small floating-point errors from producing
    # a positive value for b - a.
    difference = tf.minimum(
        b - a,
        tf.cast(-1e-12, DTYPE)
    )

    return a + tf.math.log(
        -tf.math.expm1(difference)
    )


def log_likelihood(z):
    """
    Calculate one ordinal-logistic log-likelihood for every MCMC chain.

    Parameters
    ----------
    z : Tensor
        Shape [number_of_chains, d].

    Returns
    -------
    log_lik : Tensor
        One log-likelihood value per chain.
    """

    # Transform whitened parameters into regression coefficients
    # and ordered cutpoints.
    beta, alpha = transform_parameters(z)

    # Linear predictor:
    #
    # beta shape:    [chains, p]
    # X_train shape: [observations, p]
    # eta shape:     [chains, observations]
    eta = tf.linalg.matmul(
        beta,
        X_train_tf,
        transpose_b=True
    )

    # Cumulative-logit model:
    #
    # P(Y <= k) = sigmoid(alpha_k - eta)
    #
    # logits shape:
    # [chains, observations, K - 1]
    logits = (
        alpha[:, None, :]
        - eta[:, :, None]
    )

    # log(sigmoid(logits)) gives log cumulative probabilities.
    log_cumulative = tf.math.log_sigmoid(logits)

    # log(1 - sigmoid(logits)) is calculated stably as
    # log(sigmoid(-logits)).
    log_survival = tf.math.log_sigmoid(-logits)

    # Probability of the first category:
    #
    # P(Y = 0) = P(Y <= 0)
    first_category = log_cumulative[:, :, :1]

    if K > 2:
        # Probabilities of the middle categories:
        #
        # P(Y = k) =
        # P(Y <= k) - P(Y <= k - 1)
        middle_categories = logdiffexp(
            log_cumulative[:, :, 1:],
            log_cumulative[:, :, :-1]
        )
    else:
        # For K = 2 there are no middle categories.
        middle_categories = tf.zeros(
            shape=[
                tf.shape(z)[0],
                n_train,
                0
            ],
            dtype=DTYPE
        )

    # Probability of the last category:
    #
    # P(Y = K - 1) = 1 - P(Y <= K - 2)
    last_category = log_survival[:, :, -1:]

    # Complete log-probability tensor:
    # [chains, observations, K]
    log_probabilities = tf.concat(
        [
            first_category,
            middle_categories,
            last_category
        ],
        axis=2
    )

    # Reorder to:
    # [observations, chains, K]
    log_probabilities_by_observation = tf.transpose(
        log_probabilities,
        perm=[1, 0, 2]
    )

    # Select the log-probability corresponding to the observed
    # category for each training observation.
    #
    # The result has shape:
    # [observations, chains]
    selected_log_probabilities = tf.gather(
        log_probabilities_by_observation,
        y_train_tf,
        axis=2,
        batch_dims=1
    )

    # Sum over observations, retaining one value per chain.
    log_lik = tf.reduce_sum(
        selected_log_probabilities,
        axis=0
    )

    return log_lik

In [ ]:
# ============================================================
# 9. DEFINE THE ELLIPTICAL SLICE SAMPLER
# ============================================================

def normal_sampler_fn(seed):
    """
    Generate auxiliary Gaussian samples in whitened coordinates.

    The whitened prior is z ~ Normal(0, I).
    """

    distribution = tfd.MultivariateNormalDiag(
        loc=tf.zeros(
            [N_CHAINS, d],
            dtype=DTYPE
        ),
        scale_diag=tf.ones(
            [N_CHAINS, d],
            dtype=DTYPE
        )
    )

    return distribution.sample(seed=seed)


# Create the TensorFlow Probability Elliptical Slice Sampling kernel.
# kernel = tfp.experimental.mcmc.EllipticalSliceSampler(
#     normal_sampler_fn=normal_sampler_fn,
#     log_likelihood_fn=log_likelihood
# )

In [ ]:
# # ============================================================
# # 10. INITIALISE AND RUN MCMC
# # ============================================================

# # Initialise every chain at the prior mean in whitened coordinates.
# initial_state = tf.zeros(
#     [N_CHAINS, d],
#     dtype=DTYPE
# )

# # Run Elliptical Slice Sampling.
# print("Running the ESS.")
# z_samples = tfp.mcmc.sample_chain(
#     num_results=N_SAMPLES,
#     num_burnin_steps=N_BURNIN,
#     num_steps_between_results=0,
#     current_state=initial_state,
#     kernel=kernel,
#     trace_fn=None,
#     seed=SEED,
#     parallel_iterations=1
# )
# print("Completed running ESS.")

# # Convert posterior samples from TensorFlow to NumPy.
# #
# # Shape:
# # [samples, chains, parameters]
# z_samples = z_samples.numpy()

# print(f"Whitened sample shape: {z_samples.shape}")

In [ ]:
# ============================================================
# 11. TRANSFORM POSTERIOR SAMPLES
# ============================================================

# Samples are stored as row vectors. Therefore:
#
# theta = prior_mean + z @ L.T
# theta_samples = (
#     prior_mean[None, None, :]
#     + z_samples @ L.T
# )

# Regression coefficient samples.
#
# Shape:
# [samples, chains, p]
# beta_samples = theta_samples[:, :, :p]

# Unconstrained cutpoint parameter samples.
#
# Shape:
# [samples, chains, K - 1]
# gamma_samples = theta_samples[:, :, p:]


# def construct_ordered_cutpoints_numpy(gamma):
#     """
#     NumPy equivalent of construct_ordered_cutpoints_tf().
#     """

#     first_cutpoint = gamma[..., :1]

#     if K == 2:
#         return first_cutpoint

#     # Stable NumPy implementation of softplus(x):
#     # log(1 + exp(x)).
#     positive_increments = (
#         np.logaddexp(0.0, gamma[..., 1:])
#         + 1e-8
#     )

#     remaining_cutpoints = (
#         first_cutpoint
#         + np.cumsum(
#             positive_increments,
#             axis=-1
#         )
#     )

#     return np.concatenate(
#         [
#             first_cutpoint,
#             remaining_cutpoints
#         ],
#         axis=-1
#     )


# Convert gamma samples to ordered cutpoint samples.
# cutpoint_samples = construct_ordered_cutpoints_numpy(
#     gamma_samples
# )

# print(f"Beta sample shape: {beta_samples.shape}")
# print(f"Cutpoint sample shape: {cutpoint_samples.shape}")

In [ ]:
# # ============================================================
# # 12. POSTERIOR PARAMETER SUMMARY
# # ============================================================

# # Combine chains only for posterior summaries and prediction.
# # Keep the original chain-separated arrays for convergence diagnostics.
# beta_flat = beta_samples.reshape(-1, p)

# cutpoint_flat = cutpoint_samples.reshape(
#     -1,
#     K - 1
# )

# print("\nPOSTERIOR REGRESSION COEFFICIENTS")
# print("=" * 60)

# for j in range(p):
#     posterior_values = beta_flat[:, j]

#     print(
#         f"{predictor_columns[j]}: "
#         f"Mean = {posterior_values.mean():.4f}, "
#         f"SD = {posterior_values.std(ddof=1):.4f}, "
#         f"95% CI = "
#         f"[{np.quantile(posterior_values, 0.025):.4f}, "
#         f"{np.quantile(posterior_values, 0.975):.4f}]"
#     )


# print("\nPOSTERIOR CUTPOINTS")
# print("=" * 60)

# for j in range(K - 1):
#     posterior_values = cutpoint_flat[:, j]

#     print(
#         f"Cutpoint {j + 1}: "
#         f"Mean = {posterior_values.mean():.4f}, "
#         f"SD = {posterior_values.std(ddof=1):.4f}, "
#         f"95% CI = "
#         f"[{np.quantile(posterior_values, 0.025):.4f}, "
#         f"{np.quantile(posterior_values, 0.975):.4f}]"
#     )

In [ ]:
# # ============================================================
# # 12A. TRACE PLOTS FOR POSTERIOR MIXING
# # ============================================================

# import matplotlib.pyplot as plt


# def plot_posterior_mixing(
#     beta_samples,
#     cutpoint_samples,
#     predictor_columns
# ):
#     """
#     Plot posterior MCMC trace plots for every regression coefficient
#     and every ordered cutpoint.

#     Parameters
#     ----------
#     beta_samples : np.ndarray
#         Posterior samples of regression coefficients.
#         Shape:
#             [samples, chains, p]

#     cutpoint_samples : np.ndarray
#         Posterior samples of ordered cutpoints.
#         Shape:
#             [samples, chains, K - 1]

#     predictor_columns : list
#         Names of the predictors corresponding to beta parameters.

#     Returns
#     -------
#     None
#         Displays one trace plot for every model parameter.
#     """

#     # --------------------------------------------------------
#     # Extract dimensions
#     # --------------------------------------------------------

#     n_samples = beta_samples.shape[0]
#     n_chains = beta_samples.shape[1]
#     p = beta_samples.shape[2]

#     n_cutpoints = cutpoint_samples.shape[2]

#     # MCMC iteration numbers.
#     iterations = np.arange(1, n_samples + 1)


#     # ========================================================
#     # REGRESSION COEFFICIENT TRACE PLOTS
#     # ========================================================

#     print("\n" + "=" * 60)
#     print("POSTERIOR MIXING: REGRESSION COEFFICIENTS")
#     print("=" * 60)

#     for j in range(p):

#         plt.figure(
#             figsize=(12, 4)
#         )

#         # Plot every chain separately.
#         for chain in range(n_chains):

#             plt.plot(
#                 iterations,
#                 beta_samples[:, chain, j],
#                 linewidth=0.8,
#                 alpha=0.8,
#                 label=f"Chain {chain + 1}"
#             )

#         # Posterior mean across all chains and samples.
#         posterior_mean = np.mean(
#             beta_samples[:, :, j]
#         )

#         # Add posterior mean as a horizontal reference line.
#         plt.axhline(
#             posterior_mean,
#             color="black",
#             linestyle="--",
#             linewidth=1.5,
#             label=f"Posterior mean = {posterior_mean:.3f}"
#         )

#         plt.title(
#             f"Trace Plot: Beta - {predictor_columns[j]}"
#         )

#         plt.xlabel(
#             "MCMC Iteration"
#         )

#         plt.ylabel(
#             "Posterior Sample"
#         )

#         plt.legend()

#         plt.tight_layout()

#         plt.show()


#     # ========================================================
#     # CUTPOINT TRACE PLOTS
#     # ========================================================

#     print("\n" + "=" * 60)
#     print("POSTERIOR MIXING: ORDERED CUTPOINTS")
#     print("=" * 60)

#     for j in range(n_cutpoints):

#         plt.figure(
#             figsize=(12, 4)
#         )

#         # Plot every MCMC chain independently.
#         for chain in range(n_chains):

#             plt.plot(
#                 iterations,
#                 cutpoint_samples[:, chain, j],
#                 linewidth=0.8,
#                 alpha=0.8,
#                 label=f"Chain {chain + 1}"
#             )

#         # Posterior mean across all chains and samples.
#         posterior_mean = np.mean(
#             cutpoint_samples[:, :, j]
#         )

#         plt.axhline(
#             posterior_mean,
#             color="black",
#             linestyle="--",
#             linewidth=1.5,
#             label=f"Posterior mean = {posterior_mean:.3f}"
#         )

#         plt.title(
#             f"Trace Plot: Cutpoint {j + 1}"
#         )

#         plt.xlabel(
#             "MCMC Iteration"
#         )

#         plt.ylabel(
#             "Posterior Sample"
#         )

#         plt.legend()

#         plt.tight_layout()

#         plt.show()


# # ============================================================
# # RUN THE MIXING DIAGNOSTIC
# # ============================================================

# plot_posterior_mixing(
#     beta_samples=beta_samples,
#     cutpoint_samples=cutpoint_samples,
#     predictor_columns=predictor_columns
# )

In [ ]:
# ============================================================
# 13. MCMC CONVERGENCE DIAGNOSTICS
# ============================================================

# ArviZ expects posterior arrays in the order:
#
# [chains, draws, parameters]
# idata = az.from_dict(
#     posterior={
#         "beta": np.transpose(
#             beta_samples,
#             (1, 0, 2)
#         ),
#         "cutpoints": np.transpose(
#             cutpoint_samples,
#             (1, 0, 2)
#         )
#     },
#     coords={
#         "predictor": predictor_columns,
#         "cutpoint": np.arange(1, K)
#     },
#     dims={
#         "beta": ["predictor"],
#         "cutpoints": ["cutpoint"]
#     }
# )

# print("\nMCMC DIAGNOSTICS")
# print("=" * 60)

# diagnostic_summary = az.summary(
#     idata,
#     hdi_prob=0.95,
#     round_to=4
# )

# print(diagnostic_summary)

# Common practical checks:
#
# R-hat should generally be close to 1.00.
# Values above approximately 1.01 indicate possible non-convergence.
#
# Effective sample size should not be extremely small relative
# to the total number of retained draws.
# problematic_parameters = diagnostic_summary[
#     diagnostic_summary["r_hat"] > 1.01
# ]

# if len(problematic_parameters) > 0:
#     print(
#         "\nWARNING: Some parameters have R-hat values above 1.01."
#     )
#     print(problematic_parameters[["r_hat", "ess_bulk", "ess_tail"]])
# else:
#     print("\nAll reported R-hat values are at or below 1.01.")

In [ ]:
# ============================================================
# 14. POSTERIOR PREDICTIVE PROBABILITIES
# ============================================================

def posterior_predictive_probabilities(
    X_new,
    beta_draws,
    cutpoint_draws,
    batch_size=500
):
    """
    Calculate posterior predictive category probabilities.

    Parameters
    ----------
    X_new : ndarray
        Standardized predictor matrix with shape [observations, p].

    beta_draws : ndarray
        Regression coefficient draws with shape [draws, p].

    cutpoint_draws : ndarray
        Ordered cutpoint draws with shape [draws, K - 1].

    batch_size : int
        Number of posterior draws processed per batch. Batching reduces
        memory usage when the posterior contains many samples.

    Returns
    -------
    mean_probabilities : ndarray
        Posterior mean category probabilities with shape
        [observations, K].
    """

    X_new = np.asarray(
        X_new,
        dtype=np.float64
    )

    beta_draws = np.asarray(
        beta_draws,
        dtype=np.float64
    )

    cutpoint_draws = np.asarray(
        cutpoint_draws,
        dtype=np.float64
    )

    if X_new.ndim != 2:
        raise ValueError("X_new must be a two-dimensional array.")

    if beta_draws.ndim != 2:
        raise ValueError("beta_draws must be a two-dimensional array.")

    if cutpoint_draws.ndim != 2:
        raise ValueError(
            "cutpoint_draws must be a two-dimensional array."
        )

    if X_new.shape[1] != beta_draws.shape[1]:
        raise ValueError(
            "X_new and beta_draws have incompatible predictor dimensions."
        )

    if len(beta_draws) != len(cutpoint_draws):
        raise ValueError(
            "beta_draws and cutpoint_draws must contain the same "
            "number of posterior draws."
        )

    n_observations = X_new.shape[0]

    # Accumulate category probabilities over posterior draws.
    probability_sum = np.zeros(
        (n_observations, K),
        dtype=np.float64
    )

    total_draws = len(beta_draws)

    for start in range(0, total_draws, batch_size):
        stop = min(
            start + batch_size,
            total_draws
        )

        beta_batch = beta_draws[start:stop]
        cutpoint_batch = cutpoint_draws[start:stop]

        # Linear predictors:
        # [batch draws, observations]
        eta = beta_batch @ X_new.T

        # Cumulative logits:
        # [batch draws, observations, K - 1]
        logits = (
            cutpoint_batch[:, None, :]
            - eta[:, :, None]
        )

        # Cumulative probabilities:
        #
        # P(Y <= k)
        cumulative_probabilities = expit(logits)

        first_category = cumulative_probabilities[:, :, :1]

        if K > 2:
            # Middle category probabilities:
            #
            # P(Y = k) =
            # P(Y <= k) - P(Y <= k - 1)
            middle_categories = (
                cumulative_probabilities[:, :, 1:]
                - cumulative_probabilities[:, :, :-1]
            )
        else:
            middle_categories = np.empty(
                (
                    stop - start,
                    n_observations,
                    0
                ),
                dtype=np.float64
            )

        # Last category probability:
        #
        # P(Y = K - 1) =
        # 1 - P(Y <= K - 2)
        last_category = (
            1.0
            - cumulative_probabilities[:, :, -1:]
        )

        batch_probabilities = np.concatenate(
            [
                first_category,
                middle_categories,
                last_category
            ],
            axis=2
        )

        # Guard against tiny numerical errors.
        batch_probabilities = np.clip(
            batch_probabilities,
            0.0,
            1.0
        )

        # Renormalize probabilities so every row sums to one.
        probability_totals = batch_probabilities.sum(
            axis=2,
            keepdims=True
        )

        batch_probabilities = (
            batch_probabilities
            / probability_totals
        )

        # Sum over posterior draws in the current batch.
        probability_sum += batch_probabilities.sum(axis=0)

    # Average over all posterior draws.
    mean_probabilities = probability_sum / total_draws

    # Final normalization to protect against accumulated
    # floating-point error.
    mean_probabilities = (
        mean_probabilities
        / mean_probabilities.sum(
            axis=1,
            keepdims=True
        )
    )

    return mean_probabilities


# # Combine chains for posterior prediction.
# beta_predict = beta_flat
# cutpoint_predict = cutpoint_flat

# # Predict category probabilities for test observations.
# test_probabilities = posterior_predictive_probabilities(
#     X_new=X_test,
#     beta_draws=beta_predict,
#     cutpoint_draws=cutpoint_predict,
#     batch_size=500
# )

# # Expected shape:
# # [test observations, K]
# print(
#     "\nTest probability matrix shape:",
#     test_probabilities.shape
# )

In [ ]:
# ============================================================
# 15. PREDICTED ORDINAL CATEGORIES
# ============================================================

# Select the category with the largest posterior mean probability.
# y_pred = np.argmax(
#     test_probabilities,
#     axis=1
# )

In [ ]:
# # ============================================================
# # 16. TEST-SET PERFORMANCE
# # ============================================================

# # Standard classification accuracy.
# accuracy = accuracy_score(
#     y_test,
#     y_pred
# )

# # Macro F1 gives each ordinal category equal weight.
# macro_f1 = f1_score(
#     y_test,
#     y_pred,
#     labels=np.arange(K),
#     average="macro",
#     zero_division=0
# )

# # Because classes are coded in ordinal order, MAE measures
# # the average absolute category-distance error.
# ordinal_mae = mean_absolute_error(
#     y_test,
#     y_pred
# )

# # Multiclass probabilistic log loss.
# test_log_loss = log_loss(
#     y_test,
#     test_probabilities,
#     labels=np.arange(K)
# )

# # Quadratic weightedzes larger ordinal disagreements
# # more heavily than adjacent-category disagreements.
# quadratic_kappa = cohen_kappa_score(
#     y_test,
#     y_pred,
#     labels=np.arange(K),
#     weights="quadratic"
# )

# # Create one-hot representations of the observed test outcomes.
# onehot_test = np.eye(K)[y_test]

# # Multiclass Brier score.
# multiclass_brier = np.mean(
#     np.sum(
#         (test_probabilities - onehot_test) ** 2,
#         axis=1
#     )
# )

In [ ]:
# # ============================================================
# # 17. DISPLAY TEST RESULTS
# # ============================================================

# print("\n" + "=" * 60)
# print("TEST-SET PERFORMANCE")
# print("=" * 60)

# print(f"Accuracy          : {accuracy:.4f}")
# print(f"Macro F1          : {macro_f1:.4f}")
# print(f"Ordinal MAE       : {ordinal_mae:.4f}")
# print(f"Log Loss          : {test_log_loss:.4f}")
# print(f"Quadratic Kappa   : {quadratic_kappa:.4f}")
# print(f"Multiclass Brier  : {multiclass_brier:.4f}")


# print("\nCONFUSION MATRIX")
# print("=" * 60)

# confusion = confusion_matrix(
#     y_test,
#     y_pred,
#     labels=np.arange(K)
# )

# # Display the confusion matrix with readable category labels.
# confusion_df = pd.DataFrame(
#     confusion,
#     index=[
#         f"Actual_{category}"
#         for category in category_order
#     ],
#     columns=[
#         f"Predicted_{category}"
#         for category in category_order
#     ]
# )

# print(confusion_df)


# print("\nCLASSIFICATION REPORT")
# print("=" * 60)

# print(
#     classification_report(
#         y_test,
#         y_pred,
#         labels=np.arange(K),
#         target_names=[
#             str(category)
#             for category in category_order
#         ],
#         zero_division=0
#     )
# )

In [ ]:
# # ============================================================
# # 18. STORE TEST PREDICTIONS
# # ============================================================

# # Convert integer category codes back to their original labels.
# actual_labels = category_order[y_test]
# predicted_labels = category_order[y_pred]

# prediction_results = pd.DataFrame(
#     {
#         "Actual_Code": y_test,
#         "Predicted_Code": y_pred,
#         "Actual": actual_labels,
#         "Predicted": predicted_labels,
#         "Correct": y_test == y_pred
#     }
# )

# # Add one posterior predictive probability column per category.
# for j, category in enumerate(category_order):
#     prediction_results[
#         f"Probability_{category}"
#     ] = test_probabilities[:, j]


# print("\nTEST PREDICTIONS")
# print("=" * 60)
# print(prediction_results.head(10))


# # Optional: save test predictions.
# prediction_results.to_csv(
#     "ordinal_ess_test_predictions.csv",
#     index=False
# )

# # Optional: save the MCMC diagnostic summary.
# diagnostic_summary.to_csv(
#     "ordinal_ess_mcmc_diagnostics.csv"
# )

# print(
#     "\nSaved files:"
#     "\n1. ordinal_ess_test_predictions.csv"
#     "\n2. ordinal_ess_mcmc_diagnostics.csv"
# )

## Model Iteration Multiple Train-Test Split

In [ ]:
# ============================================================
# REQUIRED IMPORTS
# ============================================================

import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ============================================================
# MULTIPLE-SEED ESS FUNCTION
# ============================================================

def run_ess_multiple_seeds(
    X,
    y,
    TEST_SIZE,
    K,
    N_CHAINS,
    N_SAMPLES,
    N_BURNIN,
    DTYPE,
    normal_sampler_fn,
    log_likelihood,
    posterior_predictive_probabilities,
    start_seed=40,
    end_seed=80,
    numerical_indices=(0, 1, 2),
    categorical_indices=(3, 4, 5),
    prediction_batch_size=500,
    output_directory=".",
    mixing_csv_name="ess_mixing_observations_all_seeds.csv",
    metrics_csv_name="ess_train_test_metrics_all_seeds.csv",
    summary_csv_name="ess_metrics_summary_all_seeds.csv"
):
    """
    Run Elliptical Slice Sampling over multiple random train-test
    split seeds.

    For every random seed, the function:

    1. Creates a stratified train-test split.
    2. Standardizes only the specified numerical predictors.
    3. Runs Elliptical Slice Sampling.
    4. Stores every retained ESS observation across all chains.
    5. Calculates train and test classification metrics.
    6. Saves one combined ESS mixing CSV.
    7. Saves one combined train-test metrics CSV.
    8. Saves one aggregate metrics-summary CSV.

    Parameters
    ----------
    X : array-like or tf.Tensor
        Predictor matrix.

    y : array-like or tf.Tensor
        Ordinal response vector. Classes must be encoded from
        0 through K - 1.

    TEST_SIZE : float
        Fraction of observations assigned to the test dataset.

    K : int
        Number of ordinal outcome categories.

    N_CHAINS : int
        Number of ESS chains.

    N_SAMPLES : int
        Number of retained posterior samples per chain.

    N_BURNIN : int
        Number of burn-in iterations.

    DTYPE : tf.DType
        TensorFlow numerical type, such as tf.float64.

    normal_sampler_fn : callable
        Normal-prior sampling function required by the
        EllipticalSliceSampler.

    log_likelihood : callable
        Log-likelihood function required by the
        EllipticalSliceSampler.

    posterior_predictive_probabilities : callable
        Function that returns posterior predictive class
        probabilities.

    start_seed : int, default=40
        First random seed, inclusive.

    end_seed : int, default=80
        Final random seed, exclusive.

    numerical_indices : tuple or list
        Indices of numerical predictors to standardize.

    categorical_indices : tuple or list
        Indices of categorical/dummy predictors that remain unchanged.

    prediction_batch_size : int, default=500
        Batch size used during posterior prediction.

    output_directory : str, default="."
        Directory in which the CSV files are saved.

    mixing_csv_name : str
        Name of the combined ESS mixing-observations CSV.

    metrics_csv_name : str
        Name of the combined train-test metrics CSV.

    summary_csv_name : str
        Name of the aggregate metrics-summary CSV.

    Returns
    -------
    results : dict
        Dictionary containing:

        results["mixing_data"]
            Combined ESS observations for every seed.

        results["metrics_data"]
            Train and test metrics for every seed.

        results["metrics_summary"]
            Aggregate metrics across all seeds.

        results["mixing_csv_path"]
            Path of the combined ESS CSV.

        results["metrics_csv_path"]
            Path of the train-test metrics CSV.

        results["summary_csv_path"]
            Path of the aggregate metrics CSV.
    """

    # ========================================================
    # 1. VALIDATE AND PREPARE OUTPUT DIRECTORY
    # ========================================================

    os.makedirs(
        output_directory,
        exist_ok=True
    )

    mixing_csv_path = os.path.join(
        output_directory,
        mixing_csv_name
    )

    metrics_csv_path = os.path.join(
        output_directory,
        metrics_csv_name
    )

    summary_csv_path = os.path.join(
        output_directory,
        summary_csv_name
    )

    if end_seed <= start_seed:
        raise ValueError(
            "end_seed must be greater than start_seed."
        )

    if K < 2:
        raise ValueError(
            "K must be at least 2."
        )

    if N_CHAINS < 1:
        raise ValueError(
            "N_CHAINS must be at least 1."
        )

    if N_SAMPLES < 1:
        raise ValueError(
            "N_SAMPLES must be at least 1."
        )

    if N_BURNIN < 0:
        raise ValueError(
            "N_BURNIN cannot be negative."
        )

    if not 0 < TEST_SIZE < 1:
        raise ValueError(
            "TEST_SIZE must be between 0 and 1."
        )

    numerical_indices = list(
        numerical_indices
    )

    categorical_indices = list(
        categorical_indices
    )

    # ========================================================
    # 2. PREPARE ORIGINAL DATA
    # ========================================================

    if tf.is_tensor(X):
        X_original = X.numpy()
    else:
        X_original = np.asarray(X)

    if tf.is_tensor(y):
        y_original = y.numpy()
    else:
        y_original = np.asarray(y)

    X_original = np.asarray(
        X_original,
        dtype=np.float64
    )

    y_original = np.asarray(
        y_original,
        dtype=np.int32
    ).reshape(-1)

    if X_original.ndim != 2:
        raise ValueError(
            "X must be a two-dimensional predictor matrix."
        )

    if y_original.ndim != 1:
        raise ValueError(
            "y must be a one-dimensional response vector."
        )

    if X_original.shape[0] != y_original.shape[0]:
        raise ValueError(
            "X and y must contain the same number of observations."
        )

    if np.isnan(X_original).any():
        raise ValueError(
            "X contains missing values."
        )

    if np.isnan(y_original.astype(np.float64)).any():
        raise ValueError(
            "y contains missing values."
        )

    unique_classes = np.unique(
        y_original
    )

    expected_classes = np.arange(
        K,
        dtype=np.int32
    )

    if not np.array_equal(
        unique_classes,
        expected_classes
    ):
        raise ValueError(
            "The response classes must be encoded as consecutive "
            f"integers from 0 through {K - 1}. "
            f"Observed classes: {unique_classes.tolist()}."
        )

    number_of_predictors = X_original.shape[1]

    all_specified_indices = (
        numerical_indices
        + categorical_indices
    )

    if len(set(all_specified_indices)) != len(all_specified_indices):
        raise ValueError(
            "numerical_indices and categorical_indices contain "
            "duplicate or overlapping column indices."
        )

    if any(
        index < 0 or index >= number_of_predictors
        for index in all_specified_indices
    ):
        raise ValueError(
            "At least one numerical or categorical column index "
            "is outside the available X columns."
        )

    # ========================================================
    # 3. INITIALIZE RESULT STORAGE
    # ========================================================

    metrics_records = []

    # Mixing observations are accumulated seed by seed.
    mixing_data_parts = []

    # ========================================================
    # 4. NUMPY ORDERED-CUTPOINT TRANSFORMATION
    # ========================================================

    def construct_ordered_cutpoints_numpy(gamma):
        """
        NumPy equivalent of the ordered-cutpoint transformation.
        """

        first_cutpoint = gamma[..., :1]

        if K == 2:
            return first_cutpoint

        positive_increments = (
            np.logaddexp(
                0.0,
                gamma[..., 1:]
            )
            + 1e-8
        )

        remaining_cutpoints = (
            first_cutpoint
            + np.cumsum(
                positive_increments,
                axis=-1
            )
        )

        ordered_cutpoints = np.concatenate(
            [
                first_cutpoint,
                remaining_cutpoints
            ],
            axis=-1
        )

        return ordered_cutpoints

    # ========================================================
    # 5. RUN ALL RANDOM-SEED ITERATIONS
    # ========================================================

    for rand_seed in range(
        start_seed,
        end_seed
    ):

        print("\n" + "=" * 70)
        print(
            f"STARTING ITERATION FOR RANDOM SEED: {rand_seed}"
        )
        print("=" * 70)

        # Reproducibility.
        np.random.seed(
            rand_seed
        )

        tf.random.set_seed(
            rand_seed
        )

        # ====================================================
        # 5.1 TRAIN-TEST SPLIT
        # ====================================================

        X_train, X_test, y_train, y_test = train_test_split(
            X_original,
            y_original,
            test_size=TEST_SIZE,
            random_state=rand_seed,
            shuffle=True,
            stratify=y_original
        )

        X_train = X_train.copy()
        X_test = X_test.copy()
        y_train = y_train.copy()
        y_test = y_test.copy()

        print("\nTrain-test split completed.")
        print("X_train shape:", X_train.shape)
        print("X_test shape :", X_test.shape)
        print("y_train shape:", y_train.shape)
        print("y_test shape :", y_test.shape)

        # ====================================================
        # 5.2 STANDARDIZE NUMERICAL PREDICTORS
        # ====================================================

        scaler = StandardScaler()

        X_train[:, numerical_indices] = scaler.fit_transform(
            X_train[:, numerical_indices]
        )

        X_test[:, numerical_indices] = scaler.transform(
            X_test[:, numerical_indices]
        )

        X_train = X_train.astype(
            np.float64,
            copy=False
        )

        X_test = X_test.astype(
            np.float64,
            copy=False
        )

        y_train = y_train.astype(
            np.int32,
            copy=False
        )

        y_test = y_test.astype(
            np.int32,
            copy=False
        )

        n_train, p = X_train.shape

        print("\nStandardization completed.")
        print(
            "Numerical predictors standardized:",
            numerical_indices
        )
        print(
            "Categorical predictors unchanged :",
            categorical_indices
        )

        # ====================================================
        # 5.3 CONVERT DATA TO TENSORFLOW TENSORS
        # ====================================================

        X_train_tf = tf.convert_to_tensor(
            X_train,
            dtype=DTYPE
        )

        y_train_tf = tf.convert_to_tensor(
            y_train,
            dtype=tf.int32
        )

        X_test_tf = tf.convert_to_tensor(
            X_test,
            dtype=DTYPE
        )

        y_test_tf = tf.convert_to_tensor(
            y_test,
            dtype=tf.int32
        )

        print("\nTensorFlow data created.")
        print(
            "X_train_tf device:",
            X_train_tf.device
        )
        print(
            "y_train_tf device:",
            y_train_tf.device
        )
        print(
            "X_test_tf device :",
            X_test_tf.device
        )
        print(
            "y_test_tf device :",
            y_test_tf.device
        )

        # These assignments ensure that a likelihood function
        # defined in the same enclosing scope can use the
        # current iteration's TensorFlow training data.
        #
        # If your log_likelihood function accepts X and y through
        # a closure or factory function, retain that existing setup.

        # ====================================================
        # 5.4 PRIOR SPECIFICATION
        # ====================================================

        d = p + K - 1

        prior_mean = np.zeros(
            d,
            dtype=np.float64
        )

        prior_sd = np.concatenate(
            [
                np.full(
                    p,
                    2.5,
                    dtype=np.float64
                ),
                np.full(
                    K - 1,
                    1.5,
                    dtype=np.float64
                )
            ]
        )

        prior_cov = np.diag(
            prior_sd ** 2
        )

        L = np.linalg.cholesky(
            prior_cov
        )

        prior_mean_tf = tf.convert_to_tensor(
            prior_mean,
            dtype=DTYPE
        )

        L_tf = tf.convert_to_tensor(
            L,
            dtype=DTYPE
        )

        print("\nPrior specification completed.")
        print(
            "Total number of unconstrained parameters:",
            d
        )

        # Keep these variables available for likelihood functions
        # that use the variables from the function's enclosing scope.
        _ = (
            n_train,
            prior_mean_tf,
            L_tf
        )

        # ====================================================
        # 5.5 CREATE ESS KERNEL
        # ====================================================

        kernel = tfp.experimental.mcmc.EllipticalSliceSampler(
            normal_sampler_fn=normal_sampler_fn,
            log_likelihood_fn=log_likelihood
        )

        # ====================================================
        # 5.6 INITIALIZE AND RUN ESS
        # ====================================================

        initial_state = tf.zeros(
            [N_CHAINS, d],
            dtype=DTYPE
        )

        print("\nRunning ESS.")

        z_samples_tf = tfp.mcmc.sample_chain(
            num_results=N_SAMPLES,
            num_burnin_steps=N_BURNIN,
            num_steps_between_results=0,
            current_state=initial_state,
            kernel=kernel,
            trace_fn=None,
            seed=rand_seed,
            parallel_iterations=1
        )

        print("Completed ESS.")

        z_samples = z_samples_tf.numpy()

        print(
            "Whitened sample shape:",
            z_samples.shape
        )

        expected_shape = (
            N_SAMPLES,
            N_CHAINS,
            d
        )

        if z_samples.shape != expected_shape:
            raise ValueError(
                "Unexpected ESS sample shape. "
                f"Expected {expected_shape}, but received "
                f"{z_samples.shape}."
            )

        # ====================================================
        # 5.7 TRANSFORM WHITENED POSTERIOR SAMPLES
        # ====================================================

        theta_samples = (
            prior_mean[None, None, :]
            + z_samples @ L.T
        )

        beta_samples = theta_samples[:, :, :p]

        gamma_samples = theta_samples[:, :, p:]

        cutpoint_samples = construct_ordered_cutpoints_numpy(
            gamma_samples
        )

        print(
            "Beta sample shape:",
            beta_samples.shape
        )

        print(
            "Gamma sample shape:",
            gamma_samples.shape
        )

        print(
            "Cutpoint sample shape:",
            cutpoint_samples.shape
        )

        # ====================================================
        # 5.8 CREATE MIXING OBSERVATIONS FOR CURRENT SEED
        # ====================================================

        sample_numbers = np.repeat(
            np.arange(
                1,
                N_SAMPLES + 1,
                dtype=np.int32
            ),
            N_CHAINS
        )

        chain_numbers = np.tile(
            np.arange(
                1,
                N_CHAINS + 1,
                dtype=np.int32
            ),
            N_SAMPLES
        )

        z_flat = z_samples.reshape(
            N_SAMPLES * N_CHAINS,
            d
        )

        theta_flat = theta_samples.reshape(
            N_SAMPLES * N_CHAINS,
            d
        )

        beta_flat = beta_samples.reshape(
            N_SAMPLES * N_CHAINS,
            p
        )

        gamma_flat = gamma_samples.reshape(
            N_SAMPLES * N_CHAINS,
            K - 1
        )

        cutpoint_flat = cutpoint_samples.reshape(
            N_SAMPLES * N_CHAINS,
            K - 1
        )

        seed_mixing_data = pd.DataFrame(
            {
                "random_seed": np.full(
                    N_SAMPLES * N_CHAINS,
                    rand_seed,
                    dtype=np.int32
                ),
                "sample": sample_numbers,
                "chain": chain_numbers
            }
        )

        # Whitened ESS state values.
        for parameter_index in range(d):
            seed_mixing_data[
                f"z_{parameter_index + 1}"
            ] = z_flat[:, parameter_index]

        # Complete transformed parameter vector.
        for parameter_index in range(d):
            seed_mixing_data[
                f"theta_{parameter_index + 1}"
            ] = theta_flat[:, parameter_index]

        # Regression coefficients.
        for parameter_index in range(p):
            seed_mixing_data[
                f"beta_{parameter_index + 1}"
            ] = beta_flat[:, parameter_index]

        # Unconstrained cutpoint parameters.
        for cutpoint_index in range(K - 1):
            seed_mixing_data[
                f"gamma_{cutpoint_index + 1}"
            ] = gamma_flat[:, cutpoint_index]

        # Ordered cutpoints.
        for cutpoint_index in range(K - 1):
            seed_mixing_data[
                f"cutpoint_{cutpoint_index + 1}"
            ] = cutpoint_flat[:, cutpoint_index]

        mixing_data_parts.append(
            seed_mixing_data
        )

        print(
            "\nESS mixing observations stored for seed:",
            rand_seed
        )

        print(
            "Mixing rows stored:",
            seed_mixing_data.shape[0]
        )

        # ====================================================
        # 5.9 COMBINE CHAINS FOR POSTERIOR PREDICTION
        # ====================================================

        beta_predict = beta_flat

        cutpoint_predict = cutpoint_flat

        # ====================================================
        # 5.10 TRAINING POSTERIOR PREDICTIONS
        # ====================================================

        train_probabilities = posterior_predictive_probabilities(
            X_new=X_train,
            beta_draws=beta_predict,
            cutpoint_draws=cutpoint_predict,
            batch_size=prediction_batch_size
        )

        train_probabilities = np.asarray(
            train_probabilities
        )

        y_pred_train = np.argmax(
            train_probabilities,
            axis=1
        ).astype(
            np.int32
        )

        # ====================================================
        # 5.11 TRAINING METRICS
        # ====================================================

        accuracy_train = accuracy_score(
            y_train,
            y_pred_train
        )

        precision_train = precision_score(
            y_train,
            y_pred_train,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        recall_train = recall_score(
            y_train,
            y_pred_train,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        f1_train = f1_score(
            y_train,
            y_pred_train,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        # ====================================================
        # 5.12 TEST POSTERIOR PREDICTIONS
        # ====================================================

        test_probabilities = posterior_predictive_probabilities(
            X_new=X_test,
            beta_draws=beta_predict,
            cutpoint_draws=cutpoint_predict,
            batch_size=prediction_batch_size
        )

        test_probabilities = np.asarray(
            test_probabilities
        )

        y_pred_test = np.argmax(
            test_probabilities,
            axis=1
        ).astype(
            np.int32
        )

        # ====================================================
        # 5.13 TEST METRICS
        # ====================================================

        accuracy_test = accuracy_score(
            y_test,
            y_pred_test
        )

        precision_test = precision_score(
            y_test,
            y_pred_test,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        recall_test = recall_score(
            y_test,
            y_pred_test,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        f1_test = f1_score(
            y_test,
            y_pred_test,
            labels=np.arange(K),
            average="macro",
            zero_division=0
        )

        # ====================================================
        # 5.14 STORE TRAIN AND TEST METRICS
        # ====================================================

        metrics_records.append(
            {
                "random_seed": rand_seed,
                "data": "train",
                "n_samples": len(y_train),
                "accuracy": accuracy_train,
                "precision": precision_train,
                "recall": recall_train,
                "f1_score": f1_train
            }
        )

        metrics_records.append(
            {
                "random_seed": rand_seed,
                "data": "test",
                "n_samples": len(y_test),
                "accuracy": accuracy_test,
                "precision": precision_test,
                "recall": recall_test,
                "f1_score": f1_test
            }
        )

        model_results = pd.DataFrame(
            [
                metrics_records[-2],
                metrics_records[-1]
            ]
        )

        # ====================================================
        # 5.15 DISPLAY CURRENT-SEED RESULTS
        # ====================================================

        print("\n" + "-" * 70)
        print(
            f"PERFORMANCE FOR RANDOM SEED: {rand_seed}"
        )
        print("-" * 70)

        print(
            model_results.to_string(
                index=False
            )
        )

        print("\n" + "=" * 70)
        print(
            f"COMPLETED ITERATION FOR RANDOM SEED: {rand_seed}"
        )
        print("=" * 70)

        # Remove large seed-specific objects before proceeding.
        del X_train_tf
        del y_train_tf
        del X_test_tf
        del y_test_tf
        del z_samples_tf
        del z_samples
        del theta_samples
        del beta_samples
        del gamma_samples
        del cutpoint_samples
        del z_flat
        del theta_flat
        del beta_flat
        del gamma_flat
        del cutpoint_flat
        del train_probabilities
        del test_probabilities

        gc.collect()

    # ========================================================
    # 6. CREATE COMBINED ESS MIXING DATAFRAME
    # ========================================================

    mixing_data = pd.concat(
        mixing_data_parts,
        axis=0,
        ignore_index=True
    )

    mixing_data.sort_values(
        by=[
            "random_seed",
            "chain",
            "sample"
        ],
        inplace=True,
        ignore_index=True
    )

    # ========================================================
    # 7. CREATE COMBINED METRICS DATAFRAME
    # ========================================================

    metrics_data = pd.DataFrame(
        metrics_records
    )

    metrics_data.sort_values(
        by=[
            "random_seed",
            "data"
        ],
        inplace=True,
        ignore_index=True
    )

    # ========================================================
    # 8. CREATE SUMMARY ACROSS ALL SEEDS
    # ========================================================

    metric_columns = [
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]

    metrics_summary = (
        metrics_data
        .groupby("data")[metric_columns]
        .agg(
            [
                "mean",
                "std",
                "min",
                "max"
            ]
        )
    )

    # Convert the MultiIndex columns into ordinary CSV columns.
    metrics_summary.columns = [
        f"{metric}_{statistic}"
        for metric, statistic in metrics_summary.columns
    ]

    metrics_summary = (
        metrics_summary
        .reset_index()
    )

    # ========================================================
    # 9. SAVE ALL RESULTS
    # ========================================================

    mixing_data.to_csv(
        mixing_csv_path,
        index=False
    )

    metrics_data.to_csv(
        metrics_csv_path,
        index=False
    )

    metrics_summary.to_csv(
        summary_csv_path,
        index=False
    )

    # ========================================================
    # 10. DISPLAY FINAL RESULTS
    # ========================================================

    total_iterations = len(
        range(
            start_seed,
            end_seed
        )
    )

    print("\n" + "=" * 70)
    print("ALL ITERATIONS COMPLETED")
    print("=" * 70)

    print(
        f"\nTotal iterations: {total_iterations}"
    )

    print(
        f"Total ESS mixing rows: {mixing_data.shape[0]}"
    )

    print(
        f"Total metric rows: {metrics_data.shape[0]}"
    )

    print("\nMetrics DataFrame:")

    print(
        metrics_data.to_string(
            index=False
        )
    )

    print("\n" + "=" * 70)
    print("AVERAGE PERFORMANCE ACROSS ALL RANDOM SEEDS")
    print("=" * 70)

    print(
        metrics_summary.to_string(
            index=False
        )
    )

    print("\n" + "=" * 70)
    print("SAVED OUTPUT FILES")
    print("=" * 70)

    print(
        "ESS mixing observations:",
        mixing_csv_path
    )

    print(
        "Train-test metrics:",
        metrics_csv_path
    )

    print(
        "Metrics summary:",
        summary_csv_path
    )

    # ========================================================
    # 11. RETURN ALL RESULTS
    # ========================================================

    results = {
        "mixing_data": mixing_data,
        "metrics_data": metrics_data,
        "metrics_summary": metrics_summary,
        "mixing_csv_path": mixing_csv_path,
        "metrics_csv_path": metrics_csv_path,
        "summary_csv_path": summary_csv_path
    }

    return results

In [ ]:
# results = run_ess_multiple_seeds(
#     X=X,
#     y=y,
#     TEST_SIZE=TEST_SIZE,
#     K=K,
#     N_CHAINS=N_CHAINS,
#     N_SAMPLES=N_SAMPLES,
#     N_BURNIN=N_BURNIN,
#     DTYPE=DTYPE,
#     normal_sampler_fn=normal_sampler_fn,
#     log_likelihood=log_likelihood,
#     posterior_predictive_probabilities=posterior_predictive_probabilities,
#     start_seed=40,
#     end_seed=50,
#     numerical_indices=[0, 1, 2],
#     categorical_indices=[3, 4, 5],
#     prediction_batch_size=500,
#     output_directory="."
# )

## Parallel Processing

In [ ]:
# ============================================================
# PARALLEL MULTI-GPU EXECUTION IMPORTS
# ============================================================

import os
import gc
import csv
import shutil
import tempfile
import numpy as np
import pandas as pd

from joblib import Parallel, delayed


# ============================================================
# CONFIGURE ONE TENSORFLOW PROCESS FOR ONE GPU
# ============================================================

# def configure_tensorflow_worker(gpu_id):
#     """
#     Assign the current worker process to one physical GPU.

#     CUDA_VISIBLE_DEVICES is set before TensorFlow performs any
#     GPU computation in the worker process.

#     After the assignment, the selected physical GPU appears as
#     logical GPU 0 inside that worker.
#     """

#     os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

#     # Prevent excessive TensorFlow console output.
#     os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

#     # Avoid unnecessary CPU-thread oversubscription when two
#     # TensorFlow workers run simultaneously.
#     os.environ["OMP_NUM_THREADS"] = "2"
#     os.environ["TF_NUM_INTRAOP_THREADS"] = "2"
#     os.environ["TF_NUM_INTEROP_THREADS"] = "1"

#     import tensorflow as tf

#     physical_gpus = tf.config.list_physical_devices("GPU")

#     if len(physical_gpus) == 0:
#         raise RuntimeError(
#             f"Worker assigned to GPU {gpu_id}, but TensorFlow "
#             "could not find a visible GPU."
#         )

#     try:
#         for physical_gpu in physical_gpus:
#             tf.config.experimental.set_memory_growth(
#                 physical_gpu,
#                 True
#             )
#     except RuntimeError as error:
#         raise RuntimeError(
#             "TensorFlow GPU memory growth must be configured "
#             "before the GPU is initialized."
#         ) from error

#     tf.config.threading.set_intra_op_parallelism_threads(2)
#     tf.config.threading.set_inter_op_parallelism_threads(1)

#     logical_gpus = tf.config.list_logical_devices("GPU")

#     print("\n" + "=" * 70)
#     print(f"CONFIGURED WORKER FOR PHYSICAL GPU: {gpu_id}")
#     print("Visible physical GPUs:", physical_gpus)
#     print("Visible logical GPUs :", logical_gpus)
#     print("=" * 70)

#     return tf


# ============================================================
# EXECUTE ONE GROUP OF SEEDS ON ONE GPU
# ============================================================

def run_seed_group_on_gpu(
    gpu_id,
    seed_group,
    X,
    y,
    TEST_SIZE,
    K,
    N_CHAINS,
    N_SAMPLES,
    N_BURNIN,
    DTYPE,
    normal_sampler_fn,
    log_likelihood,
    posterior_predictive_probabilities,
    numerical_indices,
    categorical_indices,
    prediction_batch_size,
    temporary_output_directory
):
    """
    Execute one group of random seeds sequentially on one GPU.

    Intended to be executed through Joblib ThreadingBackend.

    GPU 0 and GPU 1 are used simultaneously by two different
    worker threads, while the seeds assigned to each GPU are
    processed sequentially.
    """

    seed_group = sorted(
        list(seed_group)
    )

    if len(seed_group) == 0:
        raise ValueError(
            f"No seeds were assigned to GPU {gpu_id}."
        )

    logical_gpus = tf.config.list_logical_devices(
        "GPU"
    )

    if len(logical_gpus) == 0:
        raise RuntimeError(
            "TensorFlow did not detect any GPU."
        )

    if gpu_id >= len(logical_gpus):
        raise ValueError(
            f"Requested GPU {gpu_id}, but TensorFlow detected "
            f"only {len(logical_gpus)} GPU(s)."
        )

    device_name = f"/GPU:{gpu_id}"

    group_start_seed = seed_group[0]
    group_end_seed = seed_group[-1] + 1

    mixing_csv_name = (
        f"temporary_ess_mixing_gpu_{gpu_id}_"
        f"seeds_{group_start_seed}_{group_end_seed - 1}.csv"
    )

    metrics_csv_name = (
        f"temporary_ess_metrics_gpu_{gpu_id}_"
        f"seeds_{group_start_seed}_{group_end_seed - 1}.csv"
    )

    summary_csv_name = (
        f"temporary_ess_summary_gpu_{gpu_id}_"
        f"seeds_{group_start_seed}_{group_end_seed - 1}.csv"
    )

    print("\n" + "#" * 70)
    print(
        f"STARTING GPU WORKER {gpu_id}"
    )
    print(
        f"TensorFlow device: {device_name}"
    )
    print(
        f"Seeds assigned: {seed_group}"
    )
    print("#" * 70)

    # ========================================================
    # FORCE THIS WORKER'S TENSORFLOW OPERATIONS ONTO ITS GPU
    # ========================================================

    with tf.device(
        device_name
    ):

        worker_results = run_ess_multiple_seeds(
            X=X,
            y=y,
            TEST_SIZE=TEST_SIZE,
            K=K,
            N_CHAINS=N_CHAINS,
            N_SAMPLES=N_SAMPLES,
            N_BURNIN=N_BURNIN,
            DTYPE=DTYPE,
            normal_sampler_fn=normal_sampler_fn,
            log_likelihood=log_likelihood,
            posterior_predictive_probabilities=(
                posterior_predictive_probabilities
            ),
            start_seed=group_start_seed,
            end_seed=group_end_seed,
            numerical_indices=numerical_indices,
            categorical_indices=categorical_indices,
            prediction_batch_size=prediction_batch_size,
            output_directory=temporary_output_directory,
            mixing_csv_name=mixing_csv_name,
            metrics_csv_name=metrics_csv_name,
            summary_csv_name=summary_csv_name
        )

    worker_output = {
        "gpu_id": gpu_id,
        "device": device_name,
        "seeds": seed_group,
        "mixing_csv_path": (
            worker_results["mixing_csv_path"]
        ),
        "metrics_csv_path": (
            worker_results["metrics_csv_path"]
        ),
        "summary_csv_path": (
            worker_results["summary_csv_path"]
        )
    }

    del worker_results

    gc.collect()

    print("\n" + "#" * 70)
    print(
        f"COMPLETED GPU WORKER {gpu_id}"
    )
    print(
        f"Completed seeds: {seed_group}"
    )
    print("#" * 70)

    return worker_output

In [ ]:
# ============================================================
# COLLATE LARGE CSV FILES WITHOUT LOADING ALL ROWS INTO RAM
# ============================================================

def concatenate_csv_files(
    input_csv_paths,
    output_csv_path
):
    """
    Concatenate CSV files while writing the header only once.

    This performs streaming file collation and therefore avoids
    loading the complete ESS mixing dataset into system memory.
    """

    if len(input_csv_paths) == 0:
        raise ValueError(
            "No input CSV files were provided for collation."
        )

    with open(
        output_csv_path,
        mode="w",
        newline="",
        encoding="utf-8"
    ) as output_file:

        header_written = False

        for input_csv_path in input_csv_paths:

            with open(
                input_csv_path,
                mode="r",
                newline="",
                encoding="utf-8"
            ) as input_file:

                reader = csv.reader(
                    input_file
                )

                try:
                    header = next(
                        reader
                    )
                except StopIteration:
                    continue

                if not header_written:
                    writer = csv.writer(
                        output_file
                    )

                    writer.writerow(
                        header
                    )

                    header_written = True

                for row in reader:
                    writer.writerow(
                        row
                    )

    return output_csv_path


# ============================================================
# COLLATE METRICS AND CREATE FINAL SUMMARY
# ============================================================

def collate_metrics_files(
    metrics_csv_paths,
    final_metrics_csv_path,
    final_summary_csv_path
):
    """
    Read the small metrics files, combine them, sort by seed,
    calculate the final summary, and save both outputs.
    """

    metrics_parts = []

    for metrics_csv_path in metrics_csv_paths:

        current_metrics = pd.read_csv(
            metrics_csv_path
        )

        metrics_parts.append(
            current_metrics
        )

    metrics_data = pd.concat(
        metrics_parts,
        axis=0,
        ignore_index=True
    )

    data_sort_order = pd.Categorical(
        metrics_data["data"],
        categories=[
            "train",
            "test"
        ],
        ordered=True
    )

    metrics_data = (
        metrics_data
        .assign(
            _data_sort_order=data_sort_order
        )
        .sort_values(
            by=[
                "random_seed",
                "_data_sort_order"
            ]
        )
        .drop(
            columns="_data_sort_order"
        )
        .reset_index(
            drop=True
        )
    )

    metrics_data.to_csv(
        final_metrics_csv_path,
        index=False
    )

    metric_columns = [
        "accuracy",
        "precision",
        "recall",
        "f1_score"
    ]

    metrics_summary = (
        metrics_data
        .groupby(
            "data"
        )[metric_columns]
        .agg(
            [
                "mean",
                "std",
                "min",
                "max"
            ]
        )
    )

    metrics_summary.columns = [
        f"{metric_name}_{summary_name}"
        for metric_name, summary_name
        in metrics_summary.columns
    ]

    metrics_summary = (
        metrics_summary
        .reset_index()
    )

    summary_sort_order = pd.Categorical(
        metrics_summary["data"],
        categories=[
            "train",
            "test"
        ],
        ordered=True
    )

    metrics_summary = (
        metrics_summary
        .assign(
            _data_sort_order=summary_sort_order
        )
        .sort_values(
            "_data_sort_order"
        )
        .drop(
            columns="_data_sort_order"
        )
        .reset_index(
            drop=True
        )
    )

    metrics_summary.to_csv(
        final_summary_csv_path,
        index=False
    )

    return metrics_data, metrics_summary

In [ ]:
# ============================================================
# PARALLEL ESS ACROSS TWO GPUS
# ============================================================

def run_parallel_ess_multiple_seeds(
    X,
    y,
    TEST_SIZE,
    K,
    N_CHAINS,
    N_SAMPLES,
    N_BURNIN,
    DTYPE,
    normal_sampler_fn,
    log_likelihood,
    posterior_predictive_probabilities,
    start_seed=40,
    end_seed=80,
    numerical_indices=(0, 1, 2),
    categorical_indices=(3, 4, 5),
    prediction_batch_size=500,
    gpu_ids=(0, 1),
    output_directory=".",
    final_mixing_csv_name=(
        "ess_mixing_observations_all_seeds.csv"
    ),
    final_metrics_csv_name=(
        "ess_train_test_metrics_all_seeds.csv"
    ),
    final_summary_csv_name=(
        "ess_metrics_summary_all_seeds.csv"
    ),
    remove_temporary_files=True
):
    """
    Run independent groups of ESS seeds in parallel.

    The default configuration uses:
        GPU 0 for the first half of the seeds.
        GPU 1 for the second half of the seeds.

    Only one ESS worker is assigned to each GPU.
    """

    if end_seed <= start_seed:
        raise ValueError(
            "end_seed must be greater than start_seed."
        )

    gpu_ids = list(
        gpu_ids
    )

    if len(gpu_ids) == 0:
        raise ValueError(
            "At least one GPU ID must be supplied."
        )

    if len(set(gpu_ids)) != len(gpu_ids):
        raise ValueError(
            "gpu_ids contains duplicate GPU IDs."
        )

    os.makedirs(
        output_directory,
        exist_ok=True
    )

    all_seeds = np.arange(
        start_seed,
        end_seed,
        dtype=np.int32
    )

    number_of_workers = min(
        len(gpu_ids),
        len(all_seeds)
    )

    selected_gpu_ids = gpu_ids[
        :number_of_workers
    ]

    seed_groups = np.array_split(
        all_seeds,
        number_of_workers
    )

    seed_groups = [
        seed_group.tolist()
        for seed_group in seed_groups
        if len(seed_group) > 0
    ]

    temporary_output_directory = tempfile.mkdtemp(
        prefix="parallel_ess_",
        dir=output_directory
    )

    print("\n" + "=" * 70)
    print("PARALLEL ESS CONFIGURATION")
    print("=" * 70)

    print(
        "Selected GPUs:",
        selected_gpu_ids
    )

    print(
        "Parallel workers:",
        number_of_workers
    )

    for worker_index, seed_group in enumerate(
        seed_groups
    ):

        print(
            f"GPU {selected_gpu_ids[worker_index]} seeds:",
            seed_group
        )

    print(
        "Temporary output directory:",
        temporary_output_directory
    )

    # ========================================================
    # EXECUTE ONE LONG-RUNNING WORKER PER GPU
    # ========================================================

    worker_outputs = Parallel(
    n_jobs=number_of_workers,
    backend="threading",
    require="sharedmem",
    verbose=10
    )(
        delayed(run_seed_group_on_gpu)(
            gpu_id=selected_gpu_ids[worker_index],
            seed_group=seed_groups[worker_index],
            X=X,
            y=y,
            TEST_SIZE=TEST_SIZE,
            K=K,
            N_CHAINS=N_CHAINS,
            N_SAMPLES=N_SAMPLES,
            N_BURNIN=N_BURNIN,
            DTYPE=DTYPE,
            normal_sampler_fn=normal_sampler_fn,
            log_likelihood=log_likelihood,
            posterior_predictive_probabilities=(
                posterior_predictive_probabilities
            ),
            numerical_indices=numerical_indices,
            categorical_indices=categorical_indices,
            prediction_batch_size=prediction_batch_size,
            temporary_output_directory=(
                temporary_output_directory
            )
        )
        for worker_index in range(
            number_of_workers
        )
    )

    # Sort worker outputs according to the first assigned seed.
    worker_outputs = sorted(
        worker_outputs,
        key=lambda output: output["seeds"][0]
    )

    mixing_csv_paths = [
        output["mixing_csv_path"]
        for output in worker_outputs
    ]

    metrics_csv_paths = [
        output["metrics_csv_path"]
        for output in worker_outputs
    ]

    summary_csv_paths = [
        output["summary_csv_path"]
        for output in worker_outputs
    ]

    final_mixing_csv_path = os.path.join(
        output_directory,
        final_mixing_csv_name
    )

    final_metrics_csv_path = os.path.join(
        output_directory,
        final_metrics_csv_name
    )

    final_summary_csv_path = os.path.join(
        output_directory,
        final_summary_csv_name
    )

    # ========================================================
    # COLLATE THE LARGE MIXING CSV
    # ========================================================

    print("\nCollating ESS mixing observations.")

    concatenate_csv_files(
        input_csv_paths=mixing_csv_paths,
        output_csv_path=final_mixing_csv_path
    )

    print(
        "Combined mixing observations saved to:",
        final_mixing_csv_path
    )

    # ========================================================
    # COLLATE METRICS AND RECALCULATE THE FINAL SUMMARY
    # ========================================================

    print("\nCollating train and test metrics.")

    metrics_data, metrics_summary = collate_metrics_files(
        metrics_csv_paths=metrics_csv_paths,
        final_metrics_csv_path=final_metrics_csv_path,
        final_summary_csv_path=final_summary_csv_path
    )

    # ========================================================
    # REMOVE TEMPORARY FILES
    # ========================================================

    if remove_temporary_files:

        shutil.rmtree(
            temporary_output_directory,
            ignore_errors=True
        )

        print(
            "\nTemporary worker files removed."
        )

    # ========================================================
    # FINAL DISPLAY
    # ========================================================

    expected_mixing_rows = (
        len(all_seeds)
        * N_SAMPLES
        * N_CHAINS
    )

    expected_metric_rows = (
        len(all_seeds)
        * 2
    )

    print("\n" + "=" * 70)
    print("PARALLEL ESS COMPLETED")
    print("=" * 70)

    print(
        "Number of random seeds:",
        len(all_seeds)
    )

    print(
        "Number of parallel GPU workers:",
        number_of_workers
    )

    print(
        "Expected mixing rows:",
        expected_mixing_rows
    )

    print(
        "Expected metric rows:",
        expected_metric_rows
    )

    print(
        "Actual metric rows:",
        metrics_data.shape[0]
    )

    print("\nFinal metrics:")

    print(
        metrics_data.to_string(
            index=False
        )
    )

    print("\nFinal metrics summary:")

    print(
        metrics_summary.to_string(
            index=False
        )
    )

    print("\nSaved files:")

    print(
        "Mixing observations:",
        final_mixing_csv_path
    )

    print(
        "Train/test metrics:",
        final_metrics_csv_path
    )

    print(
        "Metrics summary:",
        final_summary_csv_path
    )

    results = {
        "metrics_data": metrics_data,
        "metrics_summary": metrics_summary,
        "mixing_csv_path": final_mixing_csv_path,
        "metrics_csv_path": final_metrics_csv_path,
        "summary_csv_path": final_summary_csv_path,
        "worker_outputs": worker_outputs,
        "number_of_workers": number_of_workers,
        "seed_groups": seed_groups
    }

    return results

In [ ]:
import inspect

source = inspect.getsource(
    run_parallel_ess_multiple_seeds
)

print(
    'backend="threading"' in source
)

print(
    'backend="loky"' in source
)

In [ ]:
worker_source = inspect.getsource(
    run_seed_group_on_gpu
)

print(
    "configure_tensorflow_worker"
    in worker_source
)

print(
    "set_memory_growth"
    in worker_source
)

In [ ]:
print("configure_tensorflow_worker" in worker_source)
print("set_memory_growth" in worker_source)

In [ ]:
gpus = tf.config.list_logical_devices(
    "GPU"
)

for i, gpu in enumerate(gpus):
    print(
        f"GPU {i}: {gpu.name}"
    )

In [ ]:
assert len(gpus) >= 2, (
    "Two GPUs are required for this parallel setup."
)

In [ ]:
from scipy.special import expit

In [ ]:
parallel_results = run_parallel_ess_multiple_seeds(
    X=X,
    y=y,
    TEST_SIZE=TEST_SIZE,
    K=K,
    N_CHAINS=N_CHAINS,
    N_SAMPLES=N_SAMPLES,
    N_BURNIN=N_BURNIN,
    DTYPE=DTYPE,
    normal_sampler_fn=normal_sampler_fn,
    log_likelihood=log_likelihood,
    posterior_predictive_probabilities=(
        posterior_predictive_probabilities
    ),
    start_seed=43,
    end_seed=51,
    numerical_indices=[0, 1, 2],
    categorical_indices=[3, 4, 5],
    prediction_batch_size=500,
    gpu_ids=[0, 1],
    output_directory=".",
    final_mixing_csv_name=(
        "ess_mixing_observations_all_seeds.csv"
    ),
    final_metrics_csv_name=(
        "ess_train_test_metrics_all_seeds.csv"
    ),
    final_summary_csv_name=(
        "ess_metrics_summary_all_seeds.csv"
    ),
    remove_temporary_files=True
)

In [ ]:
parallel_results

In [ ]:
from scipy.special import expit

test_logits = np.array(
    [-2.0, 0.0, 2.0]
)

print(
    expit(test_logits)
)

In [ ]:
print(
    "expit available:",
    "expit" in posterior_predictive_probabilities.__globals__
)

In [ ]:
metrics_data = parallel_results["metrics_data"]
metrics_summary = parallel_results["metrics_summary"]

display(metrics_data)
display(metrics_summary)

In [ ]:
# # ============================================================
# # 19. SAVE RESULTS
# # ============================================================

# metrics_data.to_csv(
#     "ordinal_ess_train_test_metrics.csv",
#     index=False
# )

# metrics_summary.to_csv(
#     "ordinal_ess_train_test_metrics_summary.csv"
# )

# print(
#     "\nSaved files:"
#     "\n1. ordinal_ess_train_test_metrics.csv"
#     "\n2. ordinal_ess_train_test_metrics_summary.csv"
# )